# Classificar a vela das 12h como: vela de alta, vela lateral, vela de baixa.

Os horários das 10h, 11h e 12h são os horários de maior volatilidade e volume. Usar os dados (preço de abertura e fechamento, preço máximo e mínimo, e volume) que compões uma vela de 10 minutos, do intervaldo das 10h às 12h para treinar uma rna com a finalidade de identificar três padrões (venda, compra, lateralidade) para o intervalo das 12h às 13h.

1. Abertura do Mercado (10:00 - 11:00)
Este é o momento mais volátil do dia. O preço de abertura é formado no leilão (09:45-10:00) e, quando a negociação começa às 10:00, o mercado absorve todas as notícias e expectativas acumuladas desde o fechamento do dia anterior. Há uma "briga" intensa entre compradores e vendedores para definir a direção inicial do dia.
Características:
* Movimentos de preço (spreads) muito amplos e rápidos.
* Volume de negociação altíssimo.

2. Abertura do Mercado Americano (11:30 - 12:30)
Por quê? O mercado brasileiro tem uma forte correlação com o mercado americano (índice S&P 500). Quando a bolsa de Nova York abre às 11:30 (horário de Brasília), há uma nova injeção de volume e volatilidade no nosso mercado. A direção que o mercado americano toma na sua primeira hora costuma influenciar fortemente o Ibovespa e, consequentemente, o mini-índice.
Características:
* Aumento súbito de volume e volatilidade.
* O mercado brasileiro pode reverter ou acelerar a tendência que vinha seguindo.
* Muitos traders aguardam essa abertura para confirmar suas posições.

O objetivo é executar uma ordem às 12h e encerrar essa ordem às 13h.

## Teste 1: Mapeamento Direto Vela-Neurônio
Testar uma rna em que os dados da primeira vela (x1 a x5) alimentão o **primeiro neurônio da camada oculta**. Os dados da segunda vela (x6 a x10) alimentão o **segundo neurônio da camada oculta**, assim por diante, os dados da ultima vela (x56 a x60) alimentão o ultimo neurônio da camada oculta.

*   **Arquitetura:**
    *   Dados da Vela 1 (x1 a x5) -> Neurônio 1 da camada oculta.
    *   Dados da Vela 2 (x6 a x10) -> Neurônio 2 da camada oculta.
    *   ...
    *   Dados da Vela 12 (x56 a x60) -> Neurônio 12 da camada oculta.
*   **Conceito:** Esta arquitetura força cada neurônio da camada oculta a se **especializar em uma única vela (um ponto específico no tempo)**. O Neurônio 1 aprenderá a extrair características apenas do que aconteceu entre 10:00 e 10:10. O Neurônio 2, do que aconteceu entre 10:10 e 10:20, e assim por diante.
*   **Vantagens:**
    *   **Interpretabilidade (relativa):** É mais fácil intuir o que cada neurônio está "olhando". Você pode analisar os pesos e ver qual das 12 velas teve mais impacto na decisão final.
    *   **Simplicidade Estrutural:** A conexão é esparsa e direta.
*   **Desvantagens/Desafios:**
    *   **Perda de Relações Cruzadas:** O modelo terá muita dificuldade em aprender padrões que envolvem a **interação entre diferentes velas**. Por exemplo, se um padrão de reversão depende da Vela 3 *e* da Vela 8, essa rede não consegue capturar essa relação na camada oculta, pois nenhum neurônio "vê" as duas velas ao mesmo tempo. A combinação das informações só ocorre na camada de saída.
    *   **Rigidez:** O modelo assume que a posição da vela no tempo é a característica mais importante. Ele não consegue identificar um mesmo padrão (ex: um engolfo de alta) se ele ocorrer às 10:30 ou às 11:10, pois seriam neurônios diferentes processando a informação.

## Teste 2: Arquitetura Totalmente Conectada (Dense)
Testar uma rna em que x1 **alimente todos os neurônios da camada oculta**, x2 também alimente todos os neurônios da camada oculta, x3 também, e assim por diante até x60.

*   **Arquitetura:**
    *   Cada feature de entrada (x1 a x60) é conectada a **todos os neurônios** da camada oculta.
*   **Conceito:** Esta é a arquitetura padrão de uma camada densa (ou *Fully Connected Layer*). Cada neurônio da camada oculta tem a capacidade de "olhar" para **todas as 60 features de entrada simultaneamente**. Ele pode aprender a dar pesos diferentes para cada uma e, assim, identificar padrões complexos que envolvem a combinação de múltiplos dados de múltiplas velas.
*   **Vantagens:**
    *   **Capacidade de Aprender Relações Complexas:** É a principal vantagem. Um neurônio pode aprender a ativar se "o volume da Vela 2 for alto E o preço de fechamento da Vela 5 romper o máximo da Vela 4". Ele pode encontrar correlações entre qualquer ponto do intervalo de 2 horas.
    *   **Flexibilidade:** É a abordagem mais comum e poderosa para dados tabulares ou vetoriais, pois não impõe restrições sobre como as features devem interagir.
*   **Desvantagens/Desafios:**
    *   **Maior Risco de Overfitting:** Com mais conexões (mais parâmetros/pesos), o modelo tem mais "liberdade" para simplesmente memorizar os dados de treino em vez de aprender padrões generalizáveis. É crucial usar técnicas de regularização (como Dropout, L1/L2).
    *   **Menos Interpretabilidade:** É mais difícil entender exatamente qual combinação de features levou a uma determinada decisão, pois todos os neurônios veem tudo.

## Qual Abordagem é Melhor?
Para o seu problema, a Arquitetura 2 (Totalmente Conectada) é teoricamente muito superior e tem uma probabilidade de sucesso drasticamente maior.
Motivo: Padrões de mercado raramente dependem de uma única vela de 10 minutos isolada. Eles são, por natureza, uma sequência de eventos. Um padrão de compra pode ser uma queda seguida por uma vela de reversão com alto volume. A Arquitetura 1 teria dificuldade em "conectar" esses dois eventos, enquanto a Arquitetura 2 foi projetada exatamente para isso.

## Etapas para o Projeto

1.  **Pré-processamento de Dados:** 
    *   **Normalização/Padronização:** É fundamental para redes neurais. Usar `MinMaxScaler` ou `StandardScaler` em todos os seus dados de entrada (Abertura, Fechamento, Máximo, Mínimo, Volume).
    *   **Criação de Features (Feature Engineering):** Em vez de usar apenas os dados brutos (OHLCV), podemos criar features mais inteligentes, como:
        *   Tamanho do corpo da vela (`abs(fechamento - abertura)`).
        *   Tamanho da sombra superior (`máximo - max(abertura, fechamento)`).
        *   Médias móveis, RSI, MACD calculados sobre as velas de 10 minutos.

2.  **Arquitetura da Rede (Sugestão para o Teste 2):**
    *   **Camada de Entrada:** 60 neurônios.
    *   **Camada(s) Oculta(s):** Começar com uma ou duas camadas densas. Um bom ponto de partida seria 64 neurônios na primeira e talvez 32 na segunda. Use a função de ativação `ReLU`.
    *   **Dropout:** Adicione camadas de Dropout (ex: `Dropout(0.3)`) após as camadas ocultas para combater o overfitting.
    *   **Camada de Saída:** 3 neurônios (um para cada classe: Compra, Venda, Lateral) com a função de ativação **`softmax`**, ideal para classificação multiclasse.
    *   **Otimizador e Perda:** Use o otimizador `Adam` e a função de perda `categorical_crossentropy`.

3.  **Validação:** Dividir os dados históricos em três conjuntos: **Treino, Validação e Teste**. Treine no primeiro, ajuste os hiperparâmetros (nº de neurônios, taxa de aprendizado, etc.) com base no desempenho do segundo e, no final, avalie o desempenho final no terceiro, que o modelo nunca viu.
 
## Exemplo

Imagine um padrão de compra clássico que se desenrola ao longo de 30 minutos:

1.  **Vela 3 (10:20-10:30):** Uma forte vela de queda, indicando pressão vendedora.
2.  **Vela 4 (10:30-10:40):** Uma vela de indecisão (um doji), com baixo volume.
3.  **Vela 5 (10:40-10:50):** Uma forte vela de alta (engolfo), com volume crescente, que "anula" a queda da Vela 3.

Este é um padrão de reversão de alta. A **condição para a compra** não está em nenhuma vela isolada, mas na **combinação sequencial** das três.

### Como Cada Arquitetura "Vê" Esse Padrão

#### Arquitetura 1 (Mapeamento Direto Vela-Neurônio)

*   **Neurônio 3** recebe os dados da Vela 3 (queda forte). Ele aprende a reconhecer "queda forte".
*   **Neurônio 4** recebe os dados da Vela 4 (doji). Ele aprende a reconhecer "indecisão".
*   **Neurônio 5** recebe os dados da Vela 5 (alta forte). Ele aprende a reconhecer "alta forte".

Até aqui, tudo bem. O problema é que **nenhum neurônio individual na camada oculta sabe que esses três eventos ocorreram em sequência**. O Neurônio 3 não sabe o que o Neurônio 5 está vendo. A única camada que recebe os sinais de todos eles ao mesmo tempo é a **camada de saída**.

Isso significa que a camada de saída (que geralmente é uma camada linear simples, com uma função softmax) tem a tarefa extremamente complexa de deduzir: "Ah, o Neurônio 3 ativou (queda), o Neurônio 4 ativou fracamente (indecisão) e o Neurônio 5 ativou fortemente (alta)... então, isso significa COMPRA".

É uma tarefa muito difícil para a camada final, pois a "inteligência" da extração de padrões complexos deveria ocorrer nas camadas ocultas. A rede perde a capacidade de criar uma "característica abstrata" que signifique "padrão de reversão de 3 velas".

#### Arquitetura 2 (Totalmente Conectada)

Nesta arquitetura, a situação é completamente diferente:

*   Um único neurônio na camada oculta (vamos chamá-lo de **Neurônio A**) recebe os dados de **TODAS** as velas (Vela 1 a 12).
*   Durante o treinamento, o **Neurônio A** pode aprender a ajustar seus pesos para se especializar em detectar exatamente o nosso padrão. Ele pode aprender a dar:
    *   **Peso alto** para as características da Vela 3 que indicam "queda forte".
    *   **Peso alto** para as características da Vela 4 que indicam "indecisão".
    *   **Peso altíssimo** para as características da Vela 5 que indicam "alta forte com volume".
    *   **Peso baixo ou zero** para todas as outras velas (1, 2, 6, 7, 8...).

Assim, o **Neurônio A** se torna um **"detector de padrão de reversão de 3 velas"**. Quando esse padrão ocorre, ele ativa fortemente e envia um sinal claro para a camada de saída que diz: "Eu encontrei um padrão de compra!". A tarefa da camada de saída fica muito mais fácil.

#### Conclusão
O ponto crucial é que a **Arquitetura 1 força cada neurônio oculto a ser míope**, vendo apenas um instante de 10 minutos. Ela delega a tarefa de conectar os pontos no tempo para a camada final, que é pouco preparada para isso.

A **Arquitetura 2 permite que os neurônios ocultos tenham uma visão panorâmica**, permitindo que eles se tornem detectores de padrões sofisticados que se desenrolam ao longo do tempo. É por isso que ela é conceitualmente mais poderosa para este tipo de problema.

## Incluir mais camadas ocultas no meu modelo do teste 1, faz com que os neurônios ocultos tenham uma visão panorâmica, assim como no teste2?
**Não, infelizmente não resolve o problema fundamental da Arquitetura 1.** Adicionar mais camadas ocultas não permitirá que ela aprenda as mesmas relações complexas que a Arquitetura 2 (Totalmente Conectada) aprende.

A resposta longa explica o porquê, e isso nos leva a um conceito central em arquitetura de redes neurais: o **fluxo de informação** e o **campo receptivo** dos neurônios.

### O Problema: O Fluxo de Informação Restrito

Vamos visualizar o fluxo de informação na sua Arquitetura 1, mesmo com camadas ocultas adicionais.

**Arquitetura 1 com Uma Camada Oculta (Original):**

*   **Entrada:**
    *   Dados da Vela 1 (x1-x5)
    *   Dados da Vela 2 (x6-x10)
    *   ...
*   **Camada Oculta 1:**
    *   Neurônio Oculto 1.1 (só vê a Vela 1)
    *   Neurônio Oculto 1.2 (só vê a Vela 2)
    *   ...
*   **Camada de Saída:**
    *   Recebe sinais de todos os neurônios da Camada Oculta 1. **É o primeiro e único lugar onde a informação de diferentes velas se encontra.**

**Arquitetura 1 com Duas Camadas Ocultas (Sua Pergunta):**

Agora, vamos adicionar uma segunda camada oculta. Como as conexões seriam? A maneira mais natural de estender a Arquitetura 1 seria uma conexão totalmente conectada entre as camadas ocultas.

*   **Entrada:**
    *   Dados da Vela 1 (x1-x5)
    *   Dados da Vela 2 (x6-x10)
    *   ...
*   **Camada Oculta 1:**
    *   Neurônio Oculto 1.1 (só vê a Vela 1)
    *   Neurônio Oculto 1.2 (só vê a Vela 2)
    *   ...
*   **Camada Oculta 2:**
    *   Cada neurônio aqui (vamos chamar de Neurônio Oculto 2.1) recebe sinais de **todos** os neurônios da Camada Oculta 1.
*   **Camada de Saída:**
    *   Recebe sinais de todos os neurônios da Camada Oculta 2.

### A Falha Crítica na Lógica

À primeira vista, parece que o problema foi resolvido! O Neurônio Oculto 2.1 agora "vê" a informação de todas as velas, certo?

**Sim, mas de uma forma extremamente limitada e pré-processada.**

O Neurônio Oculto 2.1 não vê os dados brutos da Vela 1 e da Vela 2. Ele vê:
*   A **interpretação** que o Neurônio Oculto 1.1 fez da Vela 1.
*   A **interpretação** que o Neurônio Oculto 1.2 fez da Vela 2.
*   E assim por diante.

Cada neurônio da primeira camada oculta age como um **filtro isolado**. Ele processa sua própria vela e passa adiante uma única informação resumida (sua ativação). Ele já descartou a maior parte dos detalhes brutos (a relação entre Abertura e Máxima daquela vela específica, por exemplo).

A segunda camada oculta recebe apenas esses resumos. Ela não tem mais acesso aos dados originais para encontrar correlações cruzadas. É como o jogo do "telefone sem fio":

1.  **Arquitetura 2 (Totalmente Conectada):** Um detetive (neurônio oculto) tem acesso a **todas as fotos da cena do crime** (todas as 60 features) e pode procurar por pistas que conectam uma foto à outra.

2.  **Arquitetura 1 com Camadas Extras:** Existem 12 detetives juniores. Cada um olha para **uma única foto** e escreve um resumo de uma linha sobre ela. Depois, um detetive sênior (neurônio da segunda camada) lê apenas os **12 resumos** para tentar solucionar o crime. Ele perdeu toda a riqueza de detalhes das fotos originais. Ele não pode mais verificar se a sombra na Foto 3 corresponde ao objeto na Foto 8.

### Conclusão: Por que a Arquitetura 2 é Superior

A **Arquitetura 2 (Totalmente Conectada)** dá a cada neurônio da primeira camada oculta acesso total e irrestrito a **todas as 60 features de entrada**.

Isso permite que um único neurônio aprenda a função:
`ativar = f(Volume_Vela2, Fechamento_Vela5, Máxima_Vela9, ...)`

Ele pode criar um "detector de padrões" que correlaciona diretamente qualquer detalhe bruto de qualquer vela com qualquer outro detalhe bruto de qualquer outra vela.

A **Arquitetura 1**, mesmo com mais camadas, força uma hierarquia de processamento que destrói essa capacidade. A primeira camada cria resumos isolados, e as camadas subsequentes só podem trabalhar com esses resumos, nunca com os dados originais combinados.

Portanto, para atingir seu objetivo de "identificar padrões complexos que envolvem a combinação de múltiplos dados de múltiplas velas", você precisa de uma arquitetura onde os neurônios tenham um **campo receptivo** amplo desde o início, como na Arquitetura 2 ou em Redes Neurais Convolucionais (CNNs) e Recorrentes (RNNs), que são ainda mais especializadas nisso.



## Dropout
Dropout é uma das técnicas mais importantes e eficazes para treinar redes neurais, especialmente para prevenir o overfitting. A ideia, embora simples, é extremamente poderosa.

### 1. O Problema: Overfitting (Superajuste)

Primeiro, vamos entender o problema que o Dropout resolve.

Imagine que você está treinando sua RNA para prever os padrões do mercado. Durante o treinamento, a rede ajusta seus milhões de parâmetros (os pesos das conexões) para minimizar o erro nos dados históricos que você forneceu.

O **overfitting** acontece quando a rede fica **boa demais** em prever os dados de treino. Ela não aprende os padrões gerais do mercado; em vez disso, ela praticamente **memoriza** os exemplos específicos do seu conjunto de dados, incluindo o ruído e as coincidências aleatórias.

**A consequência:**
*   O modelo tem um desempenho espetacular nos dados de treino (ex: 99% de acerto).
*   Quando você o coloca para operar com dados novos, do mundo real, que ele nunca viu, o desempenho é terrível. Ele não consegue generalizar.

É como um aluno que decora o gabarito da prova em vez de aprender a matéria. Ele tira 10 na lista de exercícios, mas zera na prova final.

### 2. A Causa do Overfitting: Co-adaptação Complexa

Uma das principais causas do overfitting é um fenômeno chamado **"co-adaptação complexa"**.

Isso acontece quando os neurônios na rede neural desenvolvem uma dependência excessiva uns dos outros. Por exemplo, um neurônio pode aprender a corrigir os erros de outro neurônio. Um pequeno grupo de neurônios pode se tornar "especialista" em detectar um padrão muito específico que só apareceu nos dados de treino.

Essa colaboração excessiva torna a rede muito rígida e frágil. Se um desses neurônios "especialistas" recebe um dado ligeiramente diferente do que ele foi treinado para ver, todo o complexo sistema de dependências pode falhar.

### 3. A Solução: Dropout (Abandono)

O Dropout combate isso de uma maneira brutalmente simples: **durante cada etapa do treinamento, ele desliga (zera) aleatoriamente uma fração dos neurônios de uma camada.**

Vamos usar o seu exemplo: `Dropout(0.3)`.

*   Você coloca uma camada de `Dropout(0.3)` logo após uma camada oculta com, digamos, 100 neurônios.
*   A cada lote de dados de treino que passa pela rede, o Dropout "sorteia" e **desliga temporariamente 30% (ou seja, 30) dos neurônios** dessa camada. A saída deles se torna zero.
*   A informação flui para a próxima camada usando apenas os 70 neurônios restantes.
*   **Importante:** No próximo lote de dados, um **novo conjunto aleatório** de 30 neurônios é desligado.

### 4. Por que Isso Funciona?

1.  **Força a Independência:** Como um neurônio nunca sabe quais de seus "colegas" estarão ativos na próxima vez, ele não pode depender de nenhum deles. Cada neurônio é forçado a aprender a extrair características úteis por conta própria. Ele precisa ser mais robusto e contribuir de forma independente para o resultado final.

2.  **Cria Redes Menores e Diversificadas:** Em cada etapa de treino, o Dropout efetivamente treina uma rede neural diferente, "mais magra". Ao longo de todo o processo de treinamento, é como se você estivesse treinando uma média de milhares de redes neurais diferentes e menores. Essa "média de modelos" (ensemble) é uma técnica comprovada para melhorar a generalização e reduzir o overfitting.

3.  **Regularização:** O Dropout age como uma forma de **regularização**, que é o termo técnico para qualquer técnica que impede o modelo de se tornar complexo demais. Ele adiciona ruído ao processo de aprendizado, forçando o modelo a aprender apenas os padrões mais fortes e consistentes, ignorando as coincidências aleatórias.

**Importante:** O Dropout só é ativo durante o **treinamento**. Quando você usa o modelo para fazer previsões (`model.predict()`) ou para avaliar (`model.evaluate()`), todos os neurônios são usados automaticamente. A biblioteca cuida disso para você.

Em resumo, o Dropout é uma ferramenta indispensável que torna suas redes neurais mais robustas e generalizáveis, forçando-as a aprender de uma maneira mais inteligente e menos dependente.

## A função `softmax`
É a escolha padrão para a camada de saída em problemas de classificação multiclasse por três motivos principais e interligados:

1.  **Converte Saídas em Probabilidades Intuitivas.**
2.  **Garante que a Soma das Probabilidades seja 1.**
3.  **Funciona Perfeitamente com a Função de Perda Correta (Cross-Entropy).**

Vamos detalhar cada um desses pontos.

### O Cenário: Antes do Softmax

Imagine a sua rede neural. Após passar por todas as camadas ocultas, a última camada (antes da função de ativação) produzirá valores brutos, chamados **logits**. Para o seu problema de 3 classes (Compra, Venda, Lateralidade), você terá 3 neurônios na camada de saída, cada um produzindo um logit.

Esses logits podem ser quaisquer números reais, positivos ou negativos. Por exemplo:

*   Neurônio "Compra": `2.7`
*   Neurônio "Venda": `-1.3`
*   Neurônio "Lateralidade": `0.9`

O que esses números significam? `2.7` é o maior, então talvez a rede "pense" que é uma compra? Mas quão confiante ela está? É uma probabilidade de 2.7%? Não. Esses números brutos não são fáceis de interpretar.

É aqui que o `softmax` entra.

### 1. Converte Saídas em Probabilidades Intuitivas

A função `softmax` pega esse vetor de logits `[2.7, -1.3, 0.9]` e o transforma em um vetor de probabilidades, onde cada valor está entre 0 e 1.

A fórmula matemática para cada neurônio é:

`Probabilidade(i) = e^(logit_i) / (Soma de todos os e^(logit_j))`

*   **`e^(logit)` (Função Exponencial):** O primeiro passo é aplicar a função exponencial (`e^x`) a cada logit. Isso tem duas vantagens:
    *   **Torna todos os valores positivos:** Mesmo o logit `-1.3` se tornará um número positivo (`e^-1.3 ≈ 0.27`). Probabilidades não podem ser negativas.
    *   **Amplifica as diferenças:** A função exponencial cresce... exponencialmente! Isso significa que um logit que já é maior (como `2.7`) se tornará *muito* maior em comparação com os outros. Isso ajuda a rede a tomar uma decisão mais "confiante".

### 2. Garante que a Soma das Probabilidades seja 1

O segundo passo da fórmula é a **normalização**. Após calcular `e^(logit)` para cada neurônio, o `softmax` divide cada um desses valores pela soma de todos eles.

Isso força a soma de todas as saídas a ser exatamente **1 (ou 100%)**.

**Continuando nosso exemplo:**

1.  **Logits:** `[2.7, -1.3, 0.9]`
2.  **Aplicar `e^x`:**
    *   `e^2.7 ≈ 14.88`
    *   `e^-1.3 ≈ 0.27`
    *   `e^0.9 ≈ 2.46`
3.  **Soma Total:** `14.88 + 0.27 + 2.46 = 17.61`
4.  **Dividir pela Soma (Normalizar):**
    *   **Prob(Compra):** `14.88 / 17.61 ≈ 0.845`  (ou 84.5%)
    *   **Prob(Venda):** `0.27 / 17.61 ≈ 0.015`   (ou 1.5%)
    *   **Prob(Lateral):** `2.46 / 17.61 ≈ 0.140`   (ou 14.0%)

**Resultado Final:** `[0.845, 0.015, 0.140]`

Agora, a saída é extremamente clara e intuitiva: a rede está **84.5% confiante** de que o padrão é de **Compra**, 1.5% de que é Venda e 14.0% de que é Lateral. A soma é 100%.

### 3. Funciona Perfeitamente com a Função de Perda Cross-Entropy

Este é o ponto mais técnico, mas crucial para o treinamento.

Para treinar a rede, precisamos de uma **função de perda (loss function)** que meça o quão "errada" a previsão foi. Para classificação multiclasse, a função de perda ideal é a **Entropia Cruzada Categórica (Categorical Cross-Entropy)**.

A Entropia Cruzada funciona medindo a "distância" entre a distribuição de probabilidade que o modelo previu e a distribuição de probabilidade verdadeira (o gabarito).

*   **Previsão do Modelo (com softmax):** `[0.845, 0.015, 0.140]`
*   **Gabarito Verdadeiro (se a resposta era "Compra"):** `[1, 0, 0]`

A Entropia Cruzada calcula uma penalidade (erro) que é **muito alta** se o modelo atribui uma probabilidade baixa à classe correta (ex: se ele desse 0.01 para "Compra" quando a resposta era "Compra"). E a penalidade é **muito baixa** se ele atribui uma probabilidade alta à classe correta.

A matemática da Entropia Cruzada e do `softmax` se complementam perfeitamente, tornando o cálculo do gradiente (o ajuste dos pesos da rede) muito estável e eficiente. Elas foram projetadas para trabalhar em conjunto.

### Resumo: Por que não outras funções?

*   **Sigmoid:** É usada para classificação **binária** (sim/não, 0/1) ou **multilabel** (onde uma amostra pode pertencer a várias classes ao mesmo tempo, ex: um filme pode ser "Ação" E "Comédia"). Ela calcula a probabilidade de cada classe de forma independente, então a soma não precisa ser 1.
*   **ReLU ou Linear:** Elas não produzem probabilidades entre 0 e 1, tornando a saída não interpretável e incompatível com a função de perda de Entropia Cruzada.

Portanto, `softmax` é a escolha ideal porque transforma os logits brutos da sua rede em uma distribuição de probabilidade limpa, interpretável e matematicamente otimizada para o processo de treinamento de um classificador multiclasse.

## Função de perda (ou *loss function*) 
É talvez a parte mais importante do treinamento de uma rede neural. Ela é o "professor" que diz à rede o quão errada ela está e em que direção ela precisa se corrigir.

A **Entropia Cruzada Categórica (Categorical Cross-Entropy)** é a parceira perfeita para a `softmax` em problemas de classificação multiclasse. Vamos entender o porquê.

### 1. O Objetivo: Medir o Erro de uma Previsão de Probabilidade

Primeiro, qual é o nosso objetivo? Queremos uma função que meça o "erro" ou a "distância" entre dois objetos:

1.  **A Previsão do Modelo:** Uma distribuição de probabilidades gerada pela `softmax`. Ex: `[0.845, 0.015, 0.140]` para (Compra, Venda, Lateral).
2.  **O Gabarito Verdadeiro:** A resposta correta, também representada como uma distribuição de probabilidade. Ex: `[1, 0, 0]` (se a resposta correta era "Compra").

Não podemos simplesmente subtrair um do outro. Uma perda de `0.1` é boa ou ruim? Precisamos de uma medida mais inteligente, e é isso que a Entropia Cruzada faz.

### 2. A Intuição: Surpresa e Penalidade

A Entropia Cruzada vem da Teoria da Informação e pode ser entendida intuitivamente como uma medida de **"surpresa"**.

Imagine que você é a função de perda. Você sabe que a resposta correta é **Compra**.

*   **Cenário 1:** O modelo prevê `[0.95, 0.02, 0.03]`. Ele está 95% confiante na resposta certa. Você não fica "surpreso". A previsão foi boa. A penalidade (o erro/loss) deve ser **muito baixa**.

*   **Cenário 2:** O modelo prevê `[0.10, 0.60, 0.30]`. Ele está apenas 10% confiante na resposta certa e acha que a resposta mais provável é "Venda". Você fica muito "surpreso" com o quão errada foi a previsão. A penalidade deve ser **muito alta**.

A Entropia Cruzada formaliza matematicamente essa noção de "surpresa": **a perda é o quão surpreso você fica ao ver a probabilidade que o modelo atribuiu à classe que era, de fato, a correta.**

### 3. Como Funciona na Prática (Matemática Simplificada)

A fórmula da Entropia Cruzada para um único exemplo é:

`Loss = - ( y_compra * log(p_compra) + y_venda * log(p_venda) + y_lateral * log(p_lateral) )`

Onde:
*   `y` é o valor do gabarito verdadeiro (0 ou 1).
*   `p` é a probabilidade prevista pelo modelo (saída da softmax).
*   `log` é o logaritmo natural.

Vamos aplicar isso aos nossos cenários. Lembre-se que o gabarito é `[y_compra=1, y_venda=0, y_lateral=0]`.

Como `y_venda` e `y_lateral` são 0, a fórmula se simplifica para:

`Loss = - ( 1 * log(p_compra) ) = -log(p_compra)`

Agora, vamos ver o comportamento da função `-log(x)`:

*   Quando `x` (a probabilidade da classe correta) está perto de **1**, `log(x)` está perto de 0. A perda é **baixa**.
*   Quando `x` está perto de **0**, `log(x)` vai para o infinito negativo. Com o sinal de menos na frente, a perda se torna **infinitamente alta**.

**Aplicando aos Cenários:**

*   **Cenário 1 (Previsão Boa):** `p_compra = 0.95`.
    *   `Loss = -log(0.95) ≈ 0.05`. Uma perda muito pequena. O "professor" está satisfeito.

*   **Cenário 2 (Previsão Ruim):** `p_compra = 0.10`.
    *   `Loss = -log(0.10) ≈ 2.30`. Uma perda muito maior. O "professor" aplica uma grande penalidade.

*   **Cenário 3 (Previsão Horrível):** O modelo prevê `p_compra = 0.0001`.
    *   `Loss = -log(0.0001) ≈ 9.21`. Uma penalidade gigantesca!

Isso é exatamente o que queremos! A função de perda penaliza exponencialmente o modelo por estar confiante na resposta errada e o recompensa por estar confiante na resposta certa.

### 4. Por que é a Parceira Ideal da Softmax?

A combinação `Softmax + Cross-Entropy` é matematicamente elegante. Quando você calcula a derivada da perda em relação aos pesos da rede (o passo fundamental do backpropagation), muitas partes complexas da matemática se cancelam, resultando em um gradiente (a direção da correção) muito simples e estável.

Isso torna o treinamento mais rápido e menos propenso a problemas numéricos, como gradientes que explodem ou desaparecem.

### Resumo

| Função de Perda | Uso Típico | Como Funciona |
| :--- | :--- | :--- |
| **Mean Squared Error (MSE)** | Regressão (prever um número, como o preço) | Calcula a média dos quadrados das diferenças entre o valor real e o previsto. Penaliza grandes erros de forma quadrática. |
| **Binary Cross-Entropy** | Classificação Binária (0 ou 1) | Versão da Entropia Cruzada para apenas duas classes. Usada com a função de ativação `sigmoid`. |
| **Categorical Cross-Entropy** | Classificação Multiclasse (uma classe de várias) | Mede a "surpresa" da previsão de probabilidade do modelo em relação à classe verdadeira. Usada com a função de ativação `softmax`. |

Em suma, a Entropia Cruzada é a maneira mais eficaz de dizer à sua rede o quão errada foi sua distribuição de probabilidade, incentivando-a a atribuir a maior probabilidade possível à classe correta.

## One-Hot Encoding
Para preparar os dados de saída (labels) para o formato [1, 0, 0] (One-Hot Encoding)

### 1. O Problema: Por que não usar números simples?

A primeira ideia que pode surgir é simplesmente atribuir um número a cada categoria:

*   Compra = 0
*   Venda = 1
*   Lateralidade = 2

Isso é chamado de **Label Encoding**. Embora pareça simples, é uma **péssima ideia** para este problema, pois cria uma relação de ordem falsa que confunde a rede neural.

Ao usar `0, 1, 2`, você está implicitamente dizendo ao modelo que:
*   `Venda (1)` é maior que `Compra (0)`.
*   `Lateralidade (2)` é o dobro de `Venda (1)`.
*   A distância entre "Compra" e "Venda" é a mesma que entre "Venda" e "Lateralidade".

Isso não faz nenhum sentido! Não existe uma relação ordinal ou de magnitude entre essas categorias. O modelo tentará encontrar padrões nessa ordem numérica falsa, o que prejudicará severamente seu aprendizado.

### 2. A Solução: One-Hot Encoding

O One-Hot Encoding resolve esse problema criando um vetor binário para cada categoria. O vetor tem o mesmo tamanho do número total de classes. Ele é composto inteiramente de zeros, exceto por um único "1" na posição que corresponde à categoria específica.

**Como funciona para o seu caso (3 classes):**

1.  **Crie um vetor de tamanho 3** para representar as classes (Compra, Venda, Lateralidade).
2.  Para cada categoria, coloque um `1` na sua posição correspondente e `0` nas outras.

| Categoria (Label) | Vetor One-Hot Encoded |
| :--- | :--- |
| **Compra** | `[1, 0, 0]` |
| **Venda** | `[0, 1, 0]` |
| **Lateralidade** | `[0, 0, 1]` |

### 3. Por que Isso é Perfeito para a Rede Neural?

Agora, vamos conectar isso com o que discutimos antes. Lembre-se da saída da sua camada `softmax` e da função de perda `categorical_cross-entropy`.

*   **Saída da Softmax (Previsão):** Um vetor de probabilidades. Ex: `[0.845, 0.015, 0.140]`
*   **Gabarito One-Hot (Verdadeiro):** Um vetor binário. Ex: `[1, 0, 0]` (se a resposta correta for "Compra")

Veja como eles se alinham perfeitamente:

*   A primeira posição em ambos os vetores representa "Compra".
*   A segunda posição em ambos representa "Venda".
*   A terceira posição em ambos representa "Lateralidade".

A função de perda `categorical_cross-entropy` pode agora comparar diretamente esses dois vetores. Ela pega a previsão do modelo (`[0.845, 0.015, 0.140]`) e a compara com o gabarito (`[1, 0, 0]`) para calcular o erro. Ela "sabe" que o objetivo era maximizar o valor na primeira posição e minimizar nas outras duas.

O One-Hot Encoding transforma seu problema de classificação categórica em um formato que permite à rede prever uma distribuição de probabilidade, que é exatamente para o que a combinação `softmax` + `cross-entropy` foi projetada.


## Otimizador e Função de Perda 
São o coração e o cérebro do processo de treinamento de uma rede neural. Eles trabalham em conjunto para garantir que o modelo aprenda de forma eficiente e correta.

Vamos detalhar a função de cada um e por que a combinação **Adam + Categorical Cross-Entropy** é tão poderosa e popular.

### A Analogia: Descendo uma Montanha no Escuro

Imagine que sua rede neural está no topo de uma vasta cordilheira, e o ponto mais baixo do vale é o lugar onde o erro (a perda) é mínimo. Sua tarefa é encontrar esse ponto mais baixo. O problema é que está tudo escuro; você só consegue sentir a inclinação do chão sob seus pés.

*   **Função de Perda (Categorical Cross-Entropy):** É o **mapa da montanha**. Ela define a paisagem. Para cada ponto (combinação de pesos da rede), ela te diz a altitude (o valor do erro/loss). Um erro alto significa que você está no alto de um pico; um erro baixo significa que você está perto de um vale. Seu objetivo é chegar à menor altitude possível.

*   **Otimizador (Adam):** É a sua **estratégia para descer a montanha**. Ele decide o tamanho e a direção de cada passo que você dá para descer. Uma estratégia ruim pode te fazer descer muito devagar, ficar preso em um platô ou até mesmo pular para o outro lado do vale. Uma boa estratégia te leva ao fundo do vale de forma rápida e confiável.

### 1. Função de Perda: `categorical_crossentropy` (O Mapa)

Já discutimos isso em detalhes, mas vamos recapitular seu papel aqui:

*   **O que faz?** Mede o quão "errada" está a previsão de probabilidade da sua camada `softmax` em comparação com o gabarito verdadeiro (One-Hot Encoded).
*   **Como funciona?** Ela calcula uma "penalidade de surpresa". A penalidade é muito alta se o modelo atribui uma probabilidade baixa à classe correta e muito baixa se o modelo está confiante na resposta certa.
*   **Seu Papel no Treinamento:** Ao final de cada passagem de dados, a `categorical_cross-entropy` calcula um único número: o **valor da perda (loss)**. É esse número que o otimizador tentará minimizar. Além disso, o cálculo da perda permite que o algoritmo de *backpropagation* determine a **inclinação (gradiente)** da montanha em cada ponto, ou seja, a direção em que o erro aumenta mais rapidamente.

### 2. Otimizador: `Adam` (A Estratégia de Descida)

"Adam" (Adaptive Moment Estimation) não é apenas um otimizador; ele é considerado um dos melhores e mais robustos otimizadores de propósito geral disponíveis hoje. Ele combina as melhores ideias de dois outros otimizadores populares: **Momentum** e **RMSprop**.

Para entender o Adam, vamos ver o que ele melhora.

#### O Otimizador Mais Simples: Gradiente Descendente (SGD)

*   **Estratégia:** Em cada passo, olhe para a inclinação (gradiente) sob seus pés e dê um passo de tamanho fixo na direção oposta (para baixo).
*   **Problemas:**
    *   **Lento em Platôs:** Se o terreno for quase plano, a inclinação é pequena, e ele dá passos minúsculos, demorando uma eternidade.
    *   **Oscilação em Vales Estreitos:** Em um vale íngreme e estreito, ele pode ficar quicando de uma parede à outra, sem conseguir descer pelo centro.
    *   **Tamanho do Passo Fixo (Taxa de Aprendizado):** Escolher o tamanho do passo é crucial e difícil. Um passo pequeno demora muito; um passo grande pode pular o fundo do vale.

#### Como o Adam Melhora Isso?

O Adam é "adaptativo". Ele ajusta o tamanho do passo (a taxa de aprendizado) para cada peso da rede individualmente, com base no histórico de como eles foram atualizados. Ele faz isso mantendo duas "memórias":

**1. Momento (Momentum) - A Bola de Neve Descendo a Ladeira:**

*   **O que é?** Adam mantém uma média móvel dos **gradientes passados** (a direção dos passos).
*   **Intuição:** Se você está consistentemente descendo na mesma direção por vários passos, você ganha "inércia" (momentum) e acelera naquela direção. Isso ajuda a passar mais rápido por platôs e a suavizar as oscilações em vales estreitos. É como uma bola de neve que ganha velocidade ao rolar.

**2. RMSprop - Adaptação por Gradiente ao Quadrado:**

*   **O que é?** Adam também mantém uma média móvel dos **quadrados dos gradientes passados**.
*   **Intuição:** Essa memória informa ao otimizador o quão "variável" ou "barulhenta" tem sido a inclinação para um peso específico.
    *   Se a inclinação para um peso tem sido consistentemente pequena (terreno plano), o Adam usa essa informação para dar um **passo maior** para aquele peso, ajudando a escapar do platô.
    *   Se a inclinação para um peso tem sido muito grande e variável (vale íngreme), ele dá um **passo menor** para aquele peso, evitando quicar de um lado para o outro.

**Adam = Momentum + RMSprop**

Adam combina essas duas ideias. Ele usa o **Momento** para determinar a direção do passo (para onde ir) e o **RMSprop** para adaptar o tamanho do passo (o quão rápido ir) para cada peso individualmente.

### Por que Adam é a Escolha Padrão?

*   **Eficiente:** Geralmente converge para uma boa solução mais rápido do que outros otimizadores.
*   **Robusto:** Funciona bem em uma ampla variedade de problemas e arquiteturas de rede com pouca necessidade de ajuste de hiperparâmetros. A taxa de aprendizado padrão (`0.001`) costuma ser um ótimo ponto de partida.
*   **Adaptativo:** A capacidade de ter taxas de aprendizado diferentes para cada peso o torna muito poderoso para lidar com os milhões de parâmetros de redes neurais profundas.

## L1 e L2 
São outras duas técnicas de regularização extremamente importantes, muitas vezes usadas em conjunto com o **Dropout**. Enquanto o Dropout ataca o overfitting "desligando" neurônios, L1 e L2 o fazem penalizando os **pesos** da rede.

A ideia central é: **um modelo complexo demais (propenso a overfitting) geralmente tem pesos com valores muito grandes.** A rede aprende a dar uma importância exagerada a certas features de entrada, memorizando o ruído.

L1 e L2 combatem isso adicionando uma "multa" (penalidade) à função de perda, baseada no tamanho dos pesos da rede. Isso força o otimizador a fazer um sacrifício: ele não pode apenas minimizar o erro de previsão; ele também precisa manter os pesos da rede pequenos.

### A Analogia: Orçamento para a Complexidade

Imagine que cada peso na sua rede neural tem um "custo".
*   Um peso grande (ex: 10.5 ou -8.2) é "caro".
*   Um peso pequeno (ex: 0.01 ou -0.02) é "barato".

A regularização L1/L2 dá ao seu modelo um **orçamento de complexidade**. O modelo é incentivado a encontrar a solução mais simples (com pesos menores) que ainda consiga prever bem os dados.

### Regularização L2 (Ridge) - "Penalidade Quadrática"

**Como funciona?**
A L2 adiciona à função de perda a **soma dos quadrados de todos os pesos** da rede, multiplicada por um pequeno fator de regularização (lambda, `λ`).

`Nova Loss = Loss Original (Cross-Entropy) + λ * (soma de todos os pesos²)`

**Qual é o efeito?**
*   **Decaimento de Peso (Weight Decay):** A penalidade quadrática força os pesos a "decaírem" em direção a zero, mas sem necessariamente chegarem a zero.
*   **Distribuição de Importância:** A L2 incentiva a rede a usar um pouco de todas as suas features de entrada, distribuindo a importância entre elas. Ela prefere ter muitos pesos pequenos a ter alguns pesos muito grandes e outros zerados. O resultado é um modelo mais "suave" e menos sensível a pequenas variações nos dados de entrada.
*   **Por que funciona?** Um peso muito grande significa que a saída da rede muda drasticamente com uma pequena mudança na entrada correspondente. Isso é um sinal de instabilidade e overfitting. Ao forçar os pesos a serem pequenos, a L2 torna o modelo mais estável e robusto.

**Intuição:** A L2 é como um "imposto sobre a riqueza" para os pesos. Quanto maior o peso, maior a penalidade quadrática que ele paga. Isso desincentiva a existência de pesos excessivamente grandes.

### Regularização L1 (Lasso) - "Penalidade Absoluta"

**Como funciona?**
A L1 adiciona à função de perda a **soma dos valores absolutos de todos os pesos**, multiplicada pelo fator de regularização (`λ`).

`Nova Loss = Loss Original (Cross-Entropy) + λ * (soma de todos os |pesos|)`

**Qual é o efeito?**
*   **Esparsidade (Sparsity):** Este é o efeito mais importante e distintivo da L1. Como a penalidade é linear (não quadrática), quando um peso não é muito útil, o otimizador descobre que a maneira mais "barata" de lidar com ele é **zerá-lo completamente**.
*   **Seleção de Atributos (Feature Selection):** Ao zerar os pesos de features de entrada inúteis ou redundantes, a L1 efetivamente realiza uma **seleção automática de atributos**. A rede aprende a focar apenas nas features mais importantes, ignorando o resto.

**Intuição:** A L1 é como um "pedágio" fixo para cada peso que não é zero. Para justificar sua existência (e não ser zerado), um peso precisa contribuir significativamente para a redução do erro de previsão, caso contrário, o custo do "pedágio" não compensa.

### Tabela Comparativa: L1 vs. L2 vs. Dropout

| Característica | Regularização L1 (Lasso) | Regularização L2 (Ridge) | Dropout |
| :--- | :--- | :--- | :--- |
| **O que penaliza?** | Valor absoluto dos pesos `|w|` | Quadrado dos pesos `w²` | A presença de neurônios |
| **Efeito Principal** | **Esparsidade**. Zera pesos inúteis. | **Decaimento de Peso**. Encolhe todos os pesos. | Força a independência dos neurônios. |
| **Resultado no Modelo** | **Seleção de Atributos**. Modelo usa poucas features importantes. | Modelo mais "suave", usa todas as features de forma distribuída. | Modelo mais robusto, como uma média de muitas redes menores. |
| **Quando usar?** | Quando você suspeita que muitas das suas features de entrada são irrelevantes. | Quase sempre uma boa ideia. É a regularização mais comum. | Em redes neurais grandes e profundas, quase sempre é benéfico. |
| **Analogia** | Pedágio (custo para existir) | Imposto sobre a riqueza (custo para ser grande) | Trabalho em equipe com membros aleatórios |

### Qual usar?
*   **Comece com L2:** É a escolha mais segura e comum.
*   **Experimente L1:** Se você tem um número muito grande de features de entrada (centenas ou milhares) e suspeita que muitas são ruído.
*   **Use em conjunto com Dropout:** Elas não são mutuamente exclusivas. É muito comum usar um regularizador L2 nos pesos e uma camada de Dropout em seguida para atacar o overfitting de duas maneiras diferentes.

A escolha do fator de regularização `λ` (ex: 0.01, 0.001, 0.0001) é um hiperparâmetro que você ajusta usando o conjunto de validação, assim como a taxa de dropout ou o número de neurônios.

## Early Stopping 
É uma das formas mais intuitivas, eficazes e quase obrigatórias de regularização. Em vez de deixar o modelo treinar por um número fixo de épocas (ex: 100 épocas), o Early Stopping monitora o desempenho do modelo e o interrompe automaticamente quando ele para de melhorar.

É uma técnica pragmática que responde à pergunta: "Qual é o momento ideal para parar de treinar?"

### 1. O Problema: O Ponto Ideal de Treinamento

Quando você treina uma rede neural, o erro no conjunto de **treino** (`loss`) quase sempre continuará diminuindo a cada época. Se você treinar por tempo suficiente, ele chegará perto de zero. Isso acontece porque o modelo está ficando cada vez melhor em memorizar os dados de treino.

No entanto, o erro no conjunto de **validação** (`val_loss`) se comporta de maneira diferente. Ele representa o quão bem o modelo generaliza para dados que não viu no treino.

1.  **Fase de Aprendizagem (Underfitting):** No início, tanto a `loss` do treino quanto a `val_loss` da validação diminuem. O modelo está aprendendo os padrões fundamentais dos dados. Ele ainda é muito simples.

2.  **Ponto Ideal (Best Model):** Há um ponto em que a `val_loss` atinge seu valor mínimo. Este é o "ponto doce", onde o modelo alcançou sua melhor capacidade de generalização. Ele aprendeu os padrões reais sem começar a memorizar o ruído.

3.  **Fase de Superajuste (Overfitting):** Se o treinamento continuar após o ponto ideal, a `loss` do treino continuará caindo, mas a `val_loss` começará a **subir**. Isso é o sinal claro de overfitting. O modelo parou de aprender padrões gerais e começou a memorizar as particularidades e o ruído do conjunto de treino, o que prejudica seu desempenho em dados novos.

**O objetivo do Early Stopping é parar o treinamento exatamente nesse ponto ideal.**

### 2. Como o Early Stopping Funciona

O Early Stopping é implementado como um "callback" no Keras. Um callback é um objeto que pode realizar ações em vários estágios do treinamento (como no final de cada época).

Você configura o callback `EarlyStopping` com algumas regras simples:

*   **`monitor`**: Qual métrica ele deve observar para decidir se o modelo está melhorando? A escolha mais comum e robusta é **`'val_loss'`**.
*   **`patience` (Paciência)**: Quantas épocas o callback deve esperar *depois* de detectar que o modelo parou de melhorar, antes de realmente parar o treinamento? Isso é crucial. O processo de treinamento tem pequenas flutuações; a `val_loss` pode subir um pouco em uma época e depois voltar a cair. A paciência evita que o treinamento pare prematuramente por causa de um pequeno ruído. Um valor comum é entre 5 e 20.
*   **`mode`**: `'min'` ou `'max'`. Indica se o objetivo é minimizar ou maximizar a métrica monitorada. Como estamos monitorando a `val_loss`, queremos minimizá-la, então o modo é `'min'` (que geralmente é o padrão).
*   **`restore_best_weights`**: Este é um parâmetro extremamente útil. Se definido como `True`, quando o treinamento for interrompido, o callback não manterá os pesos da última época, mas sim restaurará os pesos do modelo da época em que a métrica monitorada (`val_loss`) atingiu seu melhor valor. Isso garante que você fique com o melhor modelo possível, e não com um que já começou a overfitar durante as épocas de "paciência".

### Vantagens do Early Stopping

1.  **Regularização Eficaz:** É uma das maneiras mais simples e eficientes de prevenir o overfitting.
2.  **Economia de Tempo e Recursos:** Você não precisa adivinhar o número ideal de épocas. Pode definir um número alto e deixar o modelo treinar apenas o necessário, economizando tempo de computação.
3.  **Simplicidade:** É muito fácil de implementar e configurar.

## Features
Em vez de usar apenas os dados brutos (OHLCV), podemos criar features mais inteligentes, como:
    *   Tamanho do corpo da vela (`abs(fechamento - abertura)`).
    *   Tamanho da sombra superior (`máximo - max(abertura, fechamento)`).
    *   Médias móveis, RSI, MACD calculados sobre as velas de 10 minutos. 
    
### 1. Médias Móveis (Moving Averages - MAs)

**O que são?**
Uma média móvel suaviza a ação do preço, calculando o preço médio de um ativo ao longo de um número específico de períodos (no seu caso, um número de velas de 10 minutos). Ela ajuda a filtrar o "ruído" do mercado e a identificar a direção da tendência predominante.

**Como usar no seu modelo?**
Você não usaria apenas uma, mas várias, para capturar tendências de curto, médio e longo prazo dentro da sua janela de 2 horas.

*   **Média Móvel Curta (ex: 3 períodos):** Calculada sobre as últimas 3 velas de 10 minutos. Ela reage muito rápido às mudanças de preço.
    *   **Feature para a RNA:** `MA_3 = (Fechamento_Vela_N + Fechamento_Vela_N-1 + Fechamento_Vela_N-2) / 3`
*   **Média Móvel Longa (ex: 9 períodos):** Calculada sobre as últimas 9 velas de 10 minutos. Ela mostra a tendência mais consolidada.
    *   **Feature para a RNA:** `MA_9 = Média dos últimos 9 fechamentos.`

**Que informação isso dá à RNA?**
*   **Cruzamento de Médias:** Se `MA_3 > MA_9`, é um sinal de que a tendência de curto prazo está ficando mais forte que a de longo prazo (sinal de compra). Se `MA_3 < MA_9`, é um sinal de venda. A RNA pode aprender a identificar esses cruzamentos.
*   **Preço vs. Média:** Se o preço de fechamento atual está muito acima da `MA_9`, indica que o ativo pode estar "esticado" ou sobrecomprado. Se estiver muito abaixo, pode estar sobrevendido.

### 2. RSI (Relative Strength Index - Índice de Força Relativa)

**O que é?**
O RSI é um oscilador de momento que mede a velocidade e a magnitude das recentes mudanças de preço para avaliar se um ativo está **sobrecomprado** (preço subiu demais, muito rápido) ou **sobrevendido** (preço caiu demais, muito rápido). O valor do RSI varia de 0 a 100.

*   **Acima de 70:** Geralmente considerado sobrecomprado (sinal de possível reversão para queda).
*   **Abaixo de 30:** Geralmente considerado sobrevendido (sinal de possível reversão para alta).

**Como usar no seu modelo?**
Você calcularia o RSI para cada vela de 10 minutos, usando um período padrão, como 14.

*   **Feature para a RNA:** `RSI_14` para a vela atual. O cálculo é um pouco mais complexo, pois envolve a média dos ganhos e a média das perdas nas últimas 14 velas, mas todas as bibliotecas de análise técnica (como `TA-Lib` ou `pandas-ta`) fazem isso automaticamente.

**Que informação isso dá à RNA?**
*   **Níveis Extremos:** A RNA pode aprender que, quando o RSI atinge valores acima de 80, por exemplo, a probabilidade de uma correção (Venda) na hora seguinte aumenta.
*   **Divergências:** Um padrão muito poderoso. Ocorre quando o preço faz uma nova máxima, mas o RSI faz uma máxima mais baixa. Isso é uma **divergência de baixa**, um forte sinal de que a força compradora está acabando. A RNA, ao analisar a sequência de preços e a sequência de RSI, pode aprender a detectar essas divergências.

### 3. MACD (Moving Average Convergence Divergence)

**O que é?**
O MACD é um indicador de momento que segue tendências. Ele mostra a relação entre duas médias móveis exponenciais (MMEs) do preço de um ativo. É composto por três elementos:

1.  **Linha MACD:** A diferença entre a MME de 12 períodos e a MME de 26 períodos.
2.  **Linha de Sinal:** Uma MME de 9 períodos da própria linha MACD.
3.  **Histograma:** A diferença entre a Linha MACD e a Linha de Sinal. O histograma é o que dá os sinais mais claros.

**Como usar no seu modelo?**
Você calcularia os três componentes para cada vela de 10 minutos.

*   **Features para a RNA:** `MACD_line`, `MACD_signal`, `MACD_hist`.

**Que informação isso dá à RNA?**
*   **Cruzamentos:** Quando a **Linha MACD cruza para cima da Linha de Sinal**, é um sinal de compra. Quando cruza para baixo, é um sinal de venda.
*   **Força do Movimento (Histograma):**
    *   Se o histograma está positivo e crescendo, a força compradora está aumentando.
    *   Se o histograma está positivo e diminuindo, a força compradora está perdendo fôlego.
    *   Se o histograma está negativo e caindo (ficando mais negativo), a força vendedora está aumentando.
    *   Se o histograma está negativo e subindo (em direção a zero), a força vendedora está diminuindo.

A RNA pode usar o valor e a inclinação do histograma como uma poderosa feature para prever a direção e a força do próximo movimento.

### Resumo Prático

Em vez de alimentar sua rede com um vetor de 60 features (12 velas * 5 dados brutos), você agora pode alimentá-la com um vetor mais rico. Para cada uma das 12 velas, você teria:

*   Abertura
*   Máxima
*   Mínima
*   Fechamento
*   Volume
*   **MA_3**
*   **MA_9**
*   **RSI_14**
*   **MACD_line**
*   **MACD_signal**
*   **MACD_hist**

Isso transforma seus dados brutos em conhecimento acionável, aumentando drasticamente a chance de a sua rede neural encontrar padrões significativos.

## Criando Labels
A forma como você define seus labels (o "gabarito") é um dos fatores mais críticos para o sucesso do seu modelo. Uma regra de labeling bem pensada captura a "intenção" do mercado de forma mais robusta.

Sua regra atual é muito boa porque já inclui uma condição de risco/recompensa: você quer um movimento de alta (`> 20 pontos`), mas não quer que ele tenha sofrido uma grande queda antes (`pavio < 10 pontos`). Isso já é bem mais inteligente do que apenas olhar o resultado final.

Vamos explorar outros indicadores e lógicas que você pode usar, sozinhos ou em combinação, para criar labels ainda mais sofisticados.

---

### 1. Indicadores Baseados na Relação Risco/Recompensa

Esta é a categoria mais importante. Um bom trade não é apenas sobre o resultado final, mas sobre a qualidade do caminho percorrido.

#### a) Relação Drawdown/Movimento (Evolução da sua ideia)

Sua ideia de medir o pavio é ótima. Podemos generalizá-la. O "drawdown" durante a operação é a maior queda que sua posição sofreu antes de (potencialmente) se recuperar.

*   **Indicador:** `Relação_Movimento_Drawdown = (Fechamento - Abertura) / (Abertura - Mínima)`
*   **Lógica para "Compra":**
    *   `Fechamento - Abertura > X pontos` (ex: 20 pontos)
    *   E `Relação_Movimento_Drawdown > 2.0` (Significa que o movimento de alta foi pelo menos o dobro do susto/drawdown que você passou. Um valor de 2:1 é uma boa relação risco/recompensa).
*   **Vantagem:** Cria um label que representa um "trade confortável", com baixo estresse.

#### b) Eficiência do Movimento

O preço foi direto ao ponto ou ficou "sambando" antes de subir?

*   **Indicador:** `Eficiencia = (Fechamento - Abertura) / (Máxima - Mínima)`
*   **Lógica para "Compra":**
    *   `Fechamento - Abertura > X pontos`
    *   E `Eficiencia > 0.7` (Significa que o movimento líquido de alta representou 70% de toda a variação da hora. Foi um movimento direcional e eficiente).
*   **Vantagem:** Identifica tendências fortes e claras, diferenciando-as de movimentos voláteis e erráticos que por acaso terminaram positivos.

---

### 2. Indicadores Baseados em Volatilidade e Contexto

A qualidade de um movimento de 20 pontos depende do contexto. 20 pontos em um dia de alta volatilidade é quase nada; em um dia parado, é um movimento significativo.

#### a) Movimento Normalizado pela Volatilidade (ATR)

O ATR (Average True Range) é um indicador clássico de volatilidade. Ele mede o "range" médio das velas recentes.

*   **Indicador:** Calcule o ATR das velas de 10 minutos no período da manhã (ex: 10:00-12:00). Vamos chamar de `ATR_manha`.
*   **Lógica para "Compra":**
    *   `Fechamento(13h) - Abertura(12h) > 0.5 * ATR_manha` (Significa que o movimento foi pelo menos metade da "energia" ou volatilidade esperada para uma vela).
*   **Vantagem:** Cria um limiar dinâmico. Em dias voláteis, o modelo precisará prever um movimento maior para ser "Compra". Em dias parados, um movimento menor já é considerado significativo. Isso torna o modelo mais adaptativo.

#### b) Rompimento de Níveis Anteriores

O movimento das 12h-13h rompeu algum nível importante estabelecido pela manhã?

*   **Indicador:** `Máxima_Manha` (a máxima entre 10:00 e 12:00) e `Mínima_Manha`.
*   **Lógica para "Compra":**
    *   `Fechamento(13h) > Máxima_Manha` (O preço não só subiu, como rompeu a resistência da manhã).
    *   E `Mínima(12h-13h) > Mínima_Manha` (O preço não violou o suporte da manhã).
*   **Vantagem:** Ensina o modelo a identificar padrões de continuação de tendência ou rompimentos de consolidação, que são conceitos de análise técnica muito poderosos.

---

### 3. Indicadores para Definir "Lateralidade"

Definir "Lateralidade" é tão importante quanto definir "Compra" ou "Venda". Uma definição ruim pode poluir seu conjunto de dados.

#### a) Regra de "Não-Movimento" (Sua regra implícita)

*   **Lógica:** Se não for "Compra" e não for "Venda", então é "Lateral".
*   **Desvantagem:** Pode ser impreciso. Um movimento que quase foi "Compra" (ex: subiu 19 pontos em vez de 20) é tratado da mesma forma que um movimento totalmente parado.

#### b) Regra de Contenção (Range-Bound)

*   **Lógica para "Lateral":**
    *   `abs(Fechamento - Abertura) < Y pontos` (ex: 15 pontos)
    *   E `(Máxima - Mínima) < Z pontos` (ex: 50 pontos).
*   **Vantagem:** Define explicitamente a lateralidade como um período de baixo movimento líquido **e** baixa volatilidade total. Isso cria uma classe mais distinta e "verdadeiramente lateral".

#### c) Regra de "Doji Horário"

*   **Lógica para "Lateral":**
    *   O preço de fechamento da hora (13:00) está muito próximo do preço de abertura (12:00).
    *   E a máxima e a mínima estão relativamente distantes da abertura/fechamento (pavio longo para ambos os lados).
*   **Vantagem:** Captura um padrão clássico de indecisão do mercado.

### Sugestão de Combinação Robusta

Você pode combinar várias dessas ideias para criar regras de labeling muito fortes.

**Exemplo de Nova Regra para "Compra":**

1.  **Movimento Mínimo Adaptativo:** `Fechamento(13h) - Abertura(12h) > 0.5 * ATR_manha`
2.  **Qualidade do Movimento:** E `(Fechamento - Abertura) / (Abertura - Mínima) > 2.0` (Relação Risco/Recompensa favorável).
3.  **Confirmação de Força:** E `Fechamento(13h) > MédiaMovel_20_periodos(12h)` (O preço terminou acima de uma média de curto prazo).

**Exemplo de Nova Regra para "Lateral":**

1.  **Movimento Líquido Baixo:** `abs(Fechamento - Abertura) < 0.2 * ATR_manha`
2.  **Contido no Range da Manhã:** E `Máxima(12h-13h) < Máxima_Manha` E `Mínima(12h-13h) > Mínima_Manha`.

Ao usar esses indicadores mais sofisticados, você não está apenas dizendo à rede "subiu" ou "desceu". Você está ensinando a ela o que é um **"bom movimento de compra"** versus um **"movimento de sorte"**, o que é uma **"tendência clara"** versus um **"ruído volátil"**. Isso aumenta drasticamente a chance de o seu modelo aprender padrões que são genuinamente lucrativos e robustos.

## Volume tickt Vs Volume real
**sim, você pode usar ambos, e isso pode ser uma estratégia muito poderosa**, mas é preciso entender o que cada um informa ao seu modelo e os potenciais riscos.

Usar ambos não é redundante. Em vez disso, você estaria fornecendo ao modelo uma visão mais rica e multidimensional da "atividade" do mercado.

Vamos analisar o que você ganha e quais cuidados deve ter.


### O que o Modelo Pode Aprender com Ambos os Volumes?

Ao ter acesso tanto ao Volume Real (`real_volume`) quanto ao Volume de Ticks (`tick_volume`), a sua Rede Neural pode aprender a identificar padrões muito mais sutis. A **relação entre os dois volumes** se torna uma feature em si mesma.

Pense no que a divergência ou convergência entre eles pode significar:

##### Cenário 1: Volume Real Alto, Volume de Ticks Baixo

*   **O que aconteceu?** Muitas ordens grandes foram executadas em poucas transações. Em vez de 100 traders negociando 1 contrato cada, talvez 2 traders tenham negociado 50 contratos cada.
*   **Interpretação para o Modelo:** Isso é um sinal clássico de **atuação de grandes players (institucionais)**. Grandes fundos ou "baleias" estão movendo o mercado. O modelo pode aprender que, quando esse padrão ocorre após uma queda, pode ser um sinal de absorção (um grande player comprando tudo), indicando uma possível reversão para alta.
*   **Feature Implícita:** `Tamanho Médio da Ordem = Volume Real / Volume de Ticks`. Um valor alto aqui significa que as ordens executadas foram, em média, muito grandes.

##### Cenário 2: Volume Real Baixo, Volume de Ticks Alto

*   **O que aconteceu?** Houve muita "briga" e atividade, mas com ordens pequenas. Muitas transações de 1, 2 ou 5 contratos.
*   **Interpretação para o Modelo:** Isso sugere uma forte **participação do varejo (traders pessoa física)** ou um mercado com muita indecisão e "ruído". O preço se move muitas vezes, mas sem um fluxo financeiro significativo por trás. O modelo pode aprender que esse padrão em um topo de mercado indica exaustão, onde o "dinheiro inteligente" já saiu e apenas o varejo está ativo.

##### Cenário 3: Ambos Altos e Correlacionados

*   **O que aconteceu?** Muita gente negociando muitos contratos.
*   **Interpretação para o Modelo:** É um sinal inequívoco de **alto interesse e convicção** no mercado. Isso pode confirmar a força de uma tendência. Um rompimento de resistência com ambos os volumes explodindo é um sinal muito mais forte do que um rompimento com baixo volume.


#### Cuidados e Considerações

1.  **Disponibilidade do `real_volume`:** O principal obstáculo é garantir que sua corretora, via MT5, forneça dados de `real_volume` de forma consistente para todo o período histórico que você está baixando. Se em alguns períodos a coluna vier zerada, isso pode introduzir um viés terrível no seu modelo. **Verifique os dados baixados!** Analise o `DataFrame` e veja se a coluna `volume_real` contém valores não-zero em todo o histórico.
2.  **Normalização:** É crucial que você normalize (usando `MinMaxScaler` ou `StandardScaler`) ambas as colunas de volume, assim como os preços. Como as escalas de `volume_ticks` e `volume_real` podem ser muito diferentes, a normalização é indispensável para que a rede neural possa compará-las adequadamente.
3.  **Multicolinearidade:** `volume_ticks` e `volume_real` são altamente correlacionados. Em modelos estatísticos clássicos (como regressão linear), isso seria um problema (multicolinearidade). Para redes neurais, isso geralmente não é um problema grave. A rede é capaz de lidar com features correlacionadas e extrair a informação útil da relação entre elas.

#### Veredito

**Usar ambos os volumes é uma excelente ideia e uma técnica avançada de feature engineering.**

Isso permite que seu modelo vá além de simplesmente medir "atividade" e comece a entender a **"qualidade" e a "natureza"** dessa atividade (varejo vs. institucional). Se os dados de `real_volume` estiverem disponíveis e forem confiáveis, eu definitivamente recomendo incluí-los no seu modelo.

## Treino, Validação e Teste

Vamos detalhar a lógica por trás da divisão em três conjuntos: **Treino, Validação e Teste**.

#### 1. Conjunto de Treino (Training Set)

*   **O que é?** A maior parte dos seus dados históricos (geralmente 60-80%).
*   **Finalidade:** Usado exclusivamente para o processo de `model.fit()`. A rede neural "olha" para esses dados (features e labels) e ajusta seus pesos internos (parâmetros) através do backpropagation para minimizar a função de perda (Cross-Entropy). É aqui que o aprendizado ocorre.
*   **Risco:** O modelo pode facilmente "decorar" (overfit) esses dados.

#### 2. Conjunto de Validação (Validation Set)

*   **O que é?** Uma porção menor dos dados (geralmente 10-20%).
*   **Finalidade:** Usado **durante** o processo de treinamento para avaliar o modelo de forma imparcial. Ao final de cada época (uma passagem completa pelo conjunto de treino), o Keras/TensorFlow usa o conjunto de validação para testar o modelo. Isso gera métricas como `val_loss` (perda na validação) e `val_accuracy` (acurácia na validação).
*   **Como você o usa?** Você monitora essas métricas de validação para tomar decisões sobre os **hiperparâmetros**:
    *   **"A `val_loss` parou de diminuir e começou a subir, enquanto a `loss` do treino continua caindo."** -> Sinal clássico de overfitting! É hora de parar o treino (Early Stopping) ou aumentar a regularização (mais Dropout).
    *   **"A `val_accuracy` está muito baixa."** -> Talvez a rede seja simples demais (poucos neurônios), ou a taxa de aprendizado esteja errada. Você **ajusta o hiperparâmetro** e treina o modelo novamente do zero.
*   **Risco:** Ao usar o conjunto de validação repetidamente para ajustar os hiperparâmetros, você pode, sutilmente, começar a "vazar" informação sobre ele para o modelo. O modelo pode acabar ficando bom em prever o conjunto de validação, mas não necessariamente dados genéricos. É por isso que precisamos do terceiro conjunto.

#### 3. Conjunto de Teste (Test Set)

*   **O que é?** Uma porção final dos dados (geralmente 10-20%) que é **completamente intocada** durante todo o processo de treinamento e ajuste.
*   **Finalidade:** Fornecer a **avaliação final e imparcial** do desempenho do seu modelo treinado e ajustado. Você só usa este conjunto **uma única vez**, no final de tudo, com a função `model.evaluate()`.
*   **Como você o usa?** Depois de ter usado o conjunto de validação para encontrar os melhores hiperparâmetros e treinar seu modelo final, você o executa no conjunto de teste. O resultado (acurácia, precisão, recall, etc.) é a sua melhor estimativa de como o modelo se comportará em dados novos no futuro.
*   **Regra de Ouro:** **NUNCA, JAMAIS** use o conjunto de teste para tomar decisões sobre como treinar ou ajustar o modelo. Se você fizer isso, ele perde seu propósito e se torna apenas um segundo conjunto de validação.

### Como Dividir os Dados (Importante para Séries Temporais!)

Para dados normais, você pode fazer uma divisão aleatória. Mas para **séries temporais** como dados de mercado, isso é um erro grave. Você não pode usar dados do futuro para prever o passado. A divisão **deve respeitar a cronologia**.

A abordagem correta é:

1.  **Ordene** todos os seus dados históricos do mais antigo para o mais recente.
2.  **Treino:** Os primeiros 70% dos dados (ex: de 2022 a meados de 2024).
3.  **Validação:** Os próximos 15% dos dados (ex: de meados de 2024 até o final de 2024).
4.  **Teste:** Os últimos 15% dos dados (ex: todo o ano de 2025).

Dessa forma, você treina com o passado, valida com o "futuro próximo" e testa com o "futuro distante", simulando exatamente como o modelo operaria na vida real.

# Outras alternativas
Excelente pergunta! Sua abordagem atual é um ótimo ponto de partida, pois é bem definida e testável. No entanto, o mercado é dinâmico, e explorar outras estratégias pode revelar abordagens mais robustas ou lucrativas.

Aqui estão algumas alternativas, organizadas por tipo de mudança, que você pode considerar para o seu projeto.

---

### Alternativa 1: Mudar o Foco do "O Quê" para o "Quando" (Modelos de Timing)

Em vez de prever a direção de um período fixo, o objetivo aqui é identificar o **momento exato** de uma oportunidade de trade, com um alvo e stop definidos.

*   **Estratégia: Previsão de Eventos de Curto Prazo**
    *   **Objetivo do Modelo:** Em vez de prever "Compra/Venda/Lateral para a próxima hora", o modelo prevê: "A probabilidade de o preço subir `X` pontos antes de cair `Y` pontos nos próximos `N` minutos é de `Z%`?".
    *   **Como Funciona:**
        1.  **Janela Deslizante:** A cada nova vela de 10 minutos (ou até 1 minuto), o modelo analisa a janela de dados mais recente (ex: as últimas 10 velas).
        2.  **Saída do Modelo:** A RNA produz uma probabilidade para um evento, como "Cruzamento de Média Móvel", "Rompimento de Máxima/Mínima" ou "Reversão de RSI".
        3.  **Execução:** Se a probabilidade prevista para um evento de "compra" ultrapassar um limiar (ex: 75%), o robô executa a ordem imediatamente.
        4.  **Gerenciamento de Risco:** A ordem já entra com um **alvo (take profit)** e um **stop loss** pré-definidos (ex: alvo de 200 pontos, stop de 100 pontos). A operação é encerrada quando um dos dois é atingido, não por um horário fixo.
    *   **Vantagens:** Mais dinâmico, captura oportunidades a qualquer momento do dia e possui um gerenciamento de risco mais claro e integrado à operação.

---

### Alternativa 2: Mudar a Arquitetura para Capturar a "Memória" do Mercado

Sua abordagem atual usa uma janela fixa. Redes Neurais Recorrentes (RNNs), especialmente LSTMs, são projetadas para entender sequências de comprimento variável e manter uma "memória" de longo prazo.

*   **Estratégia: Modelo Preditivo com LSTM (Long Short-Term Memory)**
    *   **Objetivo do Modelo:** Prever o preço (ou a direção) da **próxima vela** com base em uma sequência de velas anteriores.
    *   **Como Funciona:**
        1.  **Treinamento:** Você treina uma LSTM para, dada uma sequência de (por exemplo) 20 velas, prever as características da 21ª vela.
        2.  **Execução:** A cada 10 minutos, o robô alimenta a sequência mais recente de velas na LSTM.
        3.  **Decisão:** Se o modelo prevê que a próxima vela terá um fechamento significativamente mais alto (ex: `> 0.2%`), ele inicia uma compra. Se prevê uma queda, inicia uma venda.
        4.  **Encerramento:** A posição pode ser encerrada no final da própria vela de 10 minutos, ou mantida enquanto as previsões subsequentes continuarem a confirmar a tendência.
    *   **Vantagens:** Arquitetura teoricamente mais adequada para séries temporais. O modelo aprende a dar pesos diferentes a eventos recentes vs. antigos de forma mais natural.

---

### Alternativa 3: Mudar o Foco do Preço para a Volatilidade

Em vez de prever a direção, que é extremamente difícil, você pode prever a **volatilidade**.

*   **Estratégia: Operar Rompimentos de Volatilidade (Breakouts)**
    *   **Objetivo do Modelo:** Prever se a próxima hora será de **alta volatilidade** (grandes movimentos) ou **baixa volatilidade** (mercado lateral).
    *   **Como Funciona:**
        1.  **Treinamento:** A RNA é treinada com dados das 10:00-12:00. O "gabarito" (label) não é "Compra/Venda", mas sim "Alta Volatilidade" ou "Baixa Volatilidade" para o período das 12:00-13:00. (Você pode definir "Alta Volatilidade" como um movimento de preço > `X` pontos, por exemplo).
        2.  **Execução:** Se o modelo prevê "Alta Volatilidade", o robô pode adotar uma estratégia de **rompimento**. Ele coloca uma ordem de compra um pouco acima da máxima do período 10-12h e uma ordem de venda um pouco abaixo da mínima. A primeira que for executada cancela a outra.
        3.  **Encerramento:** A posição é gerenciada com um alvo ou um stop móvel (trailing stop).
    *   **Vantagens:** Prever a volatilidade pode ser mais fácil do que prever a direção. Essa estratégia lucra com grandes movimentos, independentemente da direção.

---

### Alternativa 4: Mudar o Paradigma para Aprendizado por Reforço (Reinforcement Learning)

Esta é a abordagem mais avançada e complexa, mas também a que mais se assemelha a como um trader humano aprende.

*   **Estratégia: Agente de RL que Aprende a Operar**
    *   **Objetivo do Modelo:** O modelo (o "agente") não prevê o futuro. Ele aprende a tomar a **melhor ação** (Comprar, Vender, Manter/Ficar de Fora) em cada passo do tempo para maximizar uma recompensa futura (o lucro).
    *   **Como Funciona:**
        1.  **Ambiente:** Você cria um ambiente de simulação do pregão.
        2.  **Agente:** O agente observa o estado do mercado (preços, indicadores, etc.).
        3.  **Ação:** A cada vela, ele toma uma ação.
        4.  **Recompensa:** Se a ação levou a um lucro, ele recebe uma recompensa positiva. Se levou a um prejuízo, recebe uma recompensa negativa (punição).
        5.  **Aprendizado:** Ao longo de milhões de simulações, o agente aprende, por tentativa e erro, uma **política** (uma estratégia) que mapeia estados do mercado a ações que maximizam a recompensa acumulada.
    *   **Vantagens:** Potencialmente a abordagem mais poderosa, pois o modelo aprende uma estratégia completa de trading, incluindo quando entrar, quando sair e quando não fazer nada.
    *   **Desvantagens:** Extremamente complexo de implementar, treinar e validar. Requer muito mais dados e poder computacional.

### Tabela Comparativa de Alternativas

| Estratégia | Foco Principal | Vantagem Principal | Complexidade |
| :--- | :--- | :--- | :--- |
| **Sua Atual** | Prever direção de um período fixo | Bem definida, fácil de testar | Média |
| **Timing de Eventos** | Prever o "quando" de uma oportunidade | Dinâmica, gerenciamento de risco integrado | Média-Alta |
| **LSTM Preditiva** | Prever a próxima vela na sequência | Arquitetura ideal para séries temporais | Alta |
| **Previsão de Volatilidade** | Prever a magnitude do movimento | Não depende de acertar a direção | Média |
| **Aprendizado por Reforço** | Aprender uma política de ações ótima | A mais completa e adaptativa | Muito Alta |

**Minha recomendação:**
Comece com sua ideia original, pois ela é um excelente projeto de aprendizado. Depois de validá-la (ou invalidá-la), a **Alternativa 1 (Timing de Eventos)** é o próximo passo mais lógico e poderoso, pois introduz um gerenciamento de risco mais sofisticado e permite que o robô opere de forma mais flexível.

# Coletando dados do mini índice

## qual ticker de contrato contínuo usar?

Essa lista mostra as diferentes maneiras que a sua plataforma (MetaTrader) constrói a série histórica contínua, que une os diferentes vencimentos (WING26, WINJ26, etc.) em um único gráfico longo.

Vamos decifrar o que cada parte significa. Um ticker de contrato contínuo aqui é formado por: **ATIVO + CRITÉRIO + AJUSTE**.

---

### 1. Critério de Rolagem: Como a plataforma decide quando pular para o próximo contrato?

Existem duas formas principais de fazer a "rolagem" (a troca do contrato que está vencendo pelo próximo):

*   **Por Vencimento (símbolo `@`)**: A plataforma usa o contrato atual (ex: WING26) até a sua data de vencimento oficial. No dia do vencimento, ela troca para o próximo contrato (WINJ26).
    *   **Tickers na imagem:** `WIN@`, `WDO@`, `WIN@D`, `WIN@N`, `WDO@D`, `WDO@N`
    *   **Característica:** Reflete o comportamento "oficial" do mercado.

*   **Por Liquidez (símbolo `$`)**: Esta é a forma mais usada por traders. A plataforma não espera até o dia do vencimento. Ela troca para o próximo contrato alguns dias antes, quando a **liquidez** (o volume de negociações) do contrato seguinte se torna maior que a do contrato atual.
    *   **Tickers na imagem:** `WIN$`, `WDO$D`, `WDO$N`, `WIN$D`, `WIN$N`
    *   **Característica:** Reflete melhor o comportamento do "dinheiro de verdade" no mercado, pois os grandes players migram para o próximo contrato antes do vencimento. **Para backtesting, este critério é geralmente preferível.**

---

### 2. Método de Ajuste: Como a plataforma lida com o "gap" de preço na rolagem?

Quando a plataforma troca de um contrato para o outro, quase sempre há uma diferença de preço entre eles (ex: o WING26 fecha a 130.000 e o WINJ26 abre a 130.500). Esse "salto" artificial de 500 pontos não foi um movimento real do mercado, foi apenas uma junção de contratos. O método de ajuste corrige isso.

*   **Ajuste Proporcional (nenhuma letra extra ou `P` em algumas plataformas)**: Ele ajusta todo o histórico para trás de forma proporcional, para que não haja nenhum "salto" visível no gráfico.
    *   **Ticker na imagem:** `WIN$` (sozinho)
    *   **Característica:** Cria um gráfico visualmente contínuo e suave, ótimo para análise gráfica. No entanto, os preços históricos que você vê **não são os preços que foram realmente negociados** no passado.

*   **Ajuste por Diferença (letra `D`)**: Ele soma ou subtrai a diferença de preço da rolagem em todo o histórico para trás.
    *   **Tickers na imagem:** `WDO$D`, `WDO@D`, `WIN$D`, `WIN@D`
    *   **Característica:** O efeito é similar ao proporcional, mas o cálculo é diferente. Também altera os preços históricos.

*   **Sem Ajustes (letra `N` de "None")**: A plataforma simplesmente junta os contratos. O "salto" (gap) de preço na rolagem fica visível no gráfico.
    *   **Tickers na imagem:** `WDO$N`, `WDO@N`, `WIN$N`, `WIN@N`
    *   **Característica:** Os preços que você vê no gráfico são os **preços reais que foram negociados em cada dia no passado**. Para um modelo de Machine Learning, que precisa ser treinado com os valores mais fiéis possíveis à realidade, **esta é quase sempre a melhor opção.** O modelo aprenderá a lidar com os gaps de rolagem como parte do comportamento do mercado.

---

### Qual Usar no seu Código?

Com base na sua imagem e no seu objetivo de treinar uma RNA, a recomendação é clara. Você quer:
1.  **Rolagem por Liquidez (`$`)**: Para refletir o comportamento real do mercado.
2.  **Sem Ajustes (`N`)**: Para usar os preços históricos reais.

Portanto, o ticker ideal para você usar no seu script Python é:

**`WIN$N`**

Este ticker corresponde à linha: **IBOVESPA MINI - Por Liquidez (WING26) - Sem Ajustes**.

Ao usar `WIN$N`, você estará alimentando sua rede neural com os dados mais puros e representativos da realidade histórica do mercado, o que é fundamental para a qualidade do treinamento.

In [2]:
import MetaTrader5 as mt5   # pip install MetaTrader5 pandas
import pandas as pd
from datetime import datetime, timedelta
import pytz # Para lidar com fusos horários corretamente

print("Iniciando script de coleta de dados do Mini-Índice...")

# --- 1. Conexão com o MetaTrader 5 ---
# Garante que a conexão seja estabelecida antes de prosseguir.
if not mt5.initialize():
    print("Falha na inicialização do MetaTrader 5, erro:", mt5.last_error())
    # Em um notebook, não usamos quit(), apenas interrompemos a célula.
    # quit() 
else:
    print("MetaTrader 5 inicializado com sucesso!")
    print("Versão:", mt5.version())

# --- 2. Configurações da Coleta ---

# Ativo: O nome do ativo pode variar entre corretoras. 
# 'WIN$N' ou 'WIN@N' são comuns para o contrato contínuo.
# Verifique o nome correto no seu MT5 em "Observação de Mercado".
ATIVO = 'WIN$N' 

# Timeframe: M10 (10 minutos)
TIMEFRAME = mt5.TIMEFRAME_M10

# Período: Últimos 5 anos
data_fim_req = datetime.now()
data_inicio_req = data_fim_req - timedelta(days=1*365)

# Horários de interesse (em fuso horário de São Paulo)
HORA_INICIO_COLETA = 10
HORA_FIM_COLETA = 13 # Coletará até 12:59:59

# Nome do arquivo de saída
NOME_ARQUIVO_CSV = 'dados_mini_indice_10_a_13h_ultimos_5_anos.csv'

# --- 3. Função para Coletar e Processar os Dados ---

def coletar_dados_win(ativo, timeframe, data_de, data_ate):
    """
    Coleta dados históricos do MetaTrader 5 para um ativo específico.
    """
    print(f"\nBuscando dados para {ativo} de {data_de.strftime('%Y-%m-%d')} até {data_ate.strftime('%Y-%m-%d')}...")
    
    # Faz a requisição dos dados
    rates = mt5.copy_rates_range(ativo, timeframe, data_de, data_ate)
    
    if rates is None or len(rates) == 0:
        print(f"Nenhum dado retornado para o período. Verifique o nome do ativo '{ativo}'.")
        return pd.DataFrame()
        
    print(f"{len(rates)} candles brutos coletados.")
    
    df = pd.DataFrame(rates)
    df.rename(columns={
        'time': 'datetime', 'open': 'abertura', 'high': 'maxima',
        'low': 'minima', 'close': 'fechamento', 'tick_volume': 'volume'
    }, inplace=True)
    
    # Converte a coluna 'datetime' (que está em segundos UTC) para um formato de data legível
    # e ajusta para o fuso horário de São Paulo, que é o fuso do pregão.
    df['datetime'] = pd.to_datetime(df['datetime'], unit='s', utc=True)
    df = df.set_index('datetime')
    df.index = df.index.tz_convert('America/Sao_Paulo')
    
    # Seleciona apenas as colunas de interesse
    df = df[['abertura', 'maxima', 'minima', 'fechamento', 'volume']]
    return df

# --- 4. Execução da Coleta e Filtragem ---

# Verifica se a conexão foi bem-sucedida antes de prosseguir
if mt5.terminal_info():
    # Coleta os dados brutos
    dados_completos = coletar_dados_win(ATIVO, TIMEFRAME, data_inicio_req, data_fim_req)

    if not dados_completos.empty:
        print("\nFiltrando os dados pelo horário (10:00 às 13:00)...")
        
        # Filtra os dados para incluir apenas os candles entre 10:00 e 12:59
        # O método between_time já lida com os dias úteis, pois só há dados nesses dias.
        dados_filtrados = dados_completos.between_time(
            start_time=f'{HORA_INICIO_COLETA}:00', 
            end_time=f'{HORA_FIM_COLETA-1}:59'
        )
        
        print(f"{len(dados_filtrados)} candles restaram após o filtro de horário.")

        # --- 5. Salvando em CSV ---
        try:
            print(f"\nSalvando os dados filtrados no arquivo '{NOME_ARQUIVO_CSV}'...")
            dados_filtrados.to_csv(NOME_ARQUIVO_CSV)
            print("Arquivo salvo com sucesso!")
            
            # Mostra as primeiras linhas do arquivo salvo para verificação
            print("\nAmostra dos dados salvos:")
            print(dados_filtrados.head())
            
        except Exception as e:
            print(f"Ocorreu um erro ao salvar o arquivo: {e}")

# --- 6. Encerramento da Conexão ---
print("\nEncerrando a conexão com o MetaTrader 5.")
mt5.shutdown()


Iniciando script de coleta de dados do Mini-Índice...
MetaTrader 5 inicializado com sucesso!
Versão: (500, 5495, '7 Jan 2026')

Buscando dados para WIN$N de 2025-01-14 até 2026-01-14...
14216 candles brutos coletados.

Filtrando os dados pelo horário (10:00 às 13:00)...
4500 candles restaram após o filtro de horário.

Salvando os dados filtrados no arquivo 'dados_mini_indice_10_a_13h_ultimos_5_anos.csv'...
Arquivo salvo com sucesso!

Amostra dos dados salvos:
                           abertura    maxima    minima  fechamento  volume
datetime                                                                   
2025-01-15 10:00:00-03:00  121615.0  121645.0  121535.0    121625.0   34924
2025-01-15 10:10:00-03:00  121630.0  121680.0  121595.0    121645.0   27065
2025-01-15 10:20:00-03:00  121650.0  121695.0  121600.0    121650.0   27231
2025-01-15 10:30:00-03:00  121650.0  121685.0  121620.0    121660.0   23811
2025-01-15 10:40:00-03:00  121660.0  121835.0  121645.0    121830.0   45934

Enc

True

In [30]:
import datetime
import pandas as pd
import os
import MetaTrader5 as mt5
from pytz import timezone  # Import the timezone from pytz

asset_list = [
'WIN$N']

# Create timezone instance - using UTC or another timezone of your choice
timezone_instance = timezone('UTC')  # or use datetime.timezone.utc

start_date = datetime.datetime(2025, 1, 1, tzinfo=timezone_instance)  # Use the instance, not the class
end_date = datetime.datetime.today().replace(tzinfo=timezone_instance)  # Use the instance here too

# Função que busca dados de ticks de determinado ativo em uma data e armazena em um arquivo csv
def getTickData(asset, date, filepath):
    utc_start = date + datetime.timedelta(hours=9)
    utc_end = date + datetime.timedelta(hours=20)

    tick_data = mt5.copy_ticks_range(asset, utc_start, utc_end, mt5.COPY_TICKS_INFO)
    if len(tick_data) != 0:
        print("Qtde de ticks: ",len(tick_data), "\t Arquivo: ", filepath)

        df = pd.DataFrame(tick_data)
        df['time']=pd.to_datetime(df['time_msc'], unit='ms')
        df.to_csv(filepath, sep=";", index=False)

# Define root_folder if not already defined
root_folder = "./"  # Change this to your desired path

# Testa se consegue conectar a aplicação do MT5
if not mt5.initialize():
    print('Conexão falhou. Código de erro: ', mt5.last_error())
    quit()
else:
    for asset in asset_list:
        # Criar pasta do ativo se não existir
        assetTickDataPath = root_folder + asset + '/'
        isExist = os.path.exists(assetTickDataPath)
        if not isExist:
            os.makedirs(assetTickDataPath)

        symbol_sel = mt5.symbol_select(asset,True)
        if not symbol_sel:
            print('Ativo não pode ser selecionado: ', asset)
            mt5.shutdown()
        else:
            hist_date = start_date
            # Percorre todos os dias desde a data início até a data fim, excluindo fim de semana
            while hist_date <= end_date:
                if hist_date.weekday() not in (5,6):
                    # Checa se já tem arquivo com dados, se não tiver contínua
                    file = str(hist_date.year) + "{:02d}".format(int(hist_date.month)) + "{:02d}".format(int(hist_date.day)) + '.csv'
                    if (not os.path.exists(assetTickDataPath + file)):
                        getTickData(asset,hist_date, assetTickDataPath + file)
                hist_date = hist_date + datetime.timedelta(days=1)

    mt5.shutdown()

TypeError: object of type 'NoneType' has no len()

In [ ]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime, timedelta

print("Iniciando script de coleta de dados do Mini-Índice...")

# --- 1. Conexão com o MetaTrader 5 ---
if not mt5.initialize():
    print("Falha na inicialização do MetaTrader 5, erro:", mt5.last_error())
else:
    print("MetaTrader 5 inicializado com sucesso!")
    print("Versão:", mt5.version())

# --- 2. Configurações da Coleta ---
ATIVO = 'WIN$N' 
TIMEFRAME = mt5.TIMEFRAME_M10

# Usamos datas "ingênuas" (naive) para a requisição
data_inicio_req = datetime(2026, 1, 1)
data_fim_req = datetime.now()

HORA_INICIO_COLETA = 10
HORA_FIM_COLETA = 13
NOME_ARQUIVO_CSV = 'dados_mini_indice_FINAL_CORRIGIDO.csv'

# --- 3. Função para Coletar e Processar os Dados (LÓGICA CORRIGIDA) ---
def coletar_dados_win(ativo, timeframe, data_de, data_ate):
    print(f"\nBuscando dados para {ativo} de {data_de.strftime('%Y-%m-%d')} até {data_ate.strftime('%Y-%m-%d')}...")

    rates = mt5.copy_rates_range(ativo, timeframe, data_de, data_ate)
    
    if rates is None or len(rates) == 0:
        print(f"Nenhum dado retornado. Verifique o ativo e o limite de histórico.")
        return pd.DataFrame()
        
    print(f"{len(rates)} candles brutos coletados.")
    
    df = pd.DataFrame(rates)
    df.rename(columns={
        'time': 'datetime', 'open': 'abertura', 'high': 'maxima',
        'low': 'minima', 'close': 'fechamento', 'tick_volume': 'volume_ticks',
        'real_volume': 'volume_real'
    }, inplace=True)
    
    # --- LÓGICA DE TEMPO TOTALMENTE REFEITA ---
    # 1. Interpreta o timestamp do MT5 como um datetime "ingênuo" (naive).
    #    NÃO assumimos mais que é UTC.
    df['datetime'] = pd.to_datetime(df['datetime'], unit='s')
    
    # 2. APLICAMOS A CORREÇÃO MANUAL: Subtraímos 3 horas para encontrar o horário real da B3.
    #    Esta é a correção que sua evidência nos mostrou ser necessária.
    df['datetime'] = df['datetime'] - timedelta(hours=3)
    
    # 3. Definimos o datetime corrigido como o índice.
    df = df.set_index('datetime')
    # -----------------------------------------
    
    df = df[['abertura', 'maxima', 'minima', 'fechamento', 'volume_ticks', 'volume_real']]
    return df

# --- 4. Execução da Coleta e Filtragem ---
if mt5.terminal_info():
    dados_completos = coletar_dados_win(ATIVO, TIMEFRAME, data_inicio_req, data_fim_req)

    if not dados_completos.empty:
        print("\nDados após correção de -3h (este deve ser o horário real da B3):")
        print(dados_completos.head(3))
        print(dados_completos.tail(3))

        print("\nFiltrando os dados pelo horário (10:00 às 12:59)...")
        
        mascara_horario = (dados_completos.index.hour >= HORA_INICIO_COLETA) & \
                          (dados_completos.index.hour < HORA_FIM_COLETA)
        
        dados_filtrados = dados_completos[mascara_horario]
        
        print(f"{len(dados_filtrados)} candles restaram após o filtro de horário.")

        # --- 5. Salvando em CSV ---
        if not dados_filtrados.empty:
            try:
                print(f"\nSalvando os dados filtrados no arquivo '{NOME_ARQUIVO_CSV}'...")
                dados_filtrados.to_csv(NOME_ARQUIVO_CSV)
                print("Arquivo salvo com sucesso!")
                
                print("\nAmostra dos dados salvos (verifique se o horário está correto):")
                print(dados_filtrados.head())
                print(dados_filtrados.tail())
            except Exception as e:
                print(f"Ocorreu um erro ao salvar o arquivo: {e}")
        else:
            print("Nenhum dado restou após a filtragem de horário.")

# --- 6. Encerramento da Conexão ---
print("\nEncerrando a conexão com o MetaTrader 5.")
mt5.shutdown()


In [20]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime, timedelta
import pytz
import os

print("Iniciando script de coleta de dados do Mini-Índice...")

# --- 1. Conexão com o MetaTrader 5 ---
if not mt5.initialize():
    print("Falha na inicialização do MetaTrader 5, erro:", mt5.last_error())
    quit()
else:
    print("MetaTrader 5 inicializado com sucesso!")
    print("Versão:", mt5.version())

# --- 2. Configurações da Coleta ---
ATIVO = 'WIN$N' 
TIMEFRAME = mt5.TIMEFRAME_M10

# Fuso horário de referência para a requisição, como no seu exemplo.
# O servidor do MT5 parece operar em UTC.
FUSO_REQUISICAO = pytz.timezone("Etc/UTC")

# Datas de início e fim do loop
data_inicio_loop = datetime(2025, 1, 1)
data_fim_loop = datetime.now()

HORA_INICIO_COLETA = 10
HORA_FIM_COLETA = 13
NOME_ARQUIVO_CSV = 'dados_mini_indice_ADAPTADO.csv'

# Lista para armazenar os DataFrames de cada dia
lista_dfs = []

# --- 3. Loop de Coleta Diária (Lógica Adaptada) ---

# Garante que o ativo está selecionado
if not mt5.symbol_select(ATIVO, True):
    print(f"Ativo não pode ser selecionado: {ATIVO}")
    mt5.shutdown()
    quit()

print(f"Iniciando loop de coleta para o ativo {ATIVO}...")
hist_date = data_inicio_loop

while hist_date <= data_fim_loop:
    # Pula fins de semana
    if hist_date.weekday() not in (5, 6):
        
        # --- LÓGICA DE TEMPO ADAPTADA DO SEU CÓDIGO ---
        # 1. Define o início e o fim do dia no fuso UTC para a requisição.
        #    Usamos um intervalo amplo (9h-20h UTC) para garantir a captura de todo o pregão da B3,
        #    considerando o deslocamento de 3 horas.
        #    (Pregão 10h-18h B3 = 13h-21h UTC)
        utc_from = FUSO_REQUISICAO.localize(datetime(hist_date.year, hist_date.month, hist_date.day, hour=9))
        utc_to = FUSO_REQUISICAO.localize(datetime(hist_date.year, hist_date.month, hist_date.day, hour=22))
        
        # 2. Faz a requisição ao MT5 com o período UTC pré-ajustado.
        rates = mt5.copy_rates_range(ATIVO, TIMEFRAME, utc_from, utc_to)
        
        if rates is not None and len(rates) > 0:
            df_dia = pd.DataFrame(rates)
            
            # 3. O timestamp retornado é interpretado como UTC e convertido para o horário da B3.
            df_dia['datetime'] = pd.to_datetime(df_dia['time'], unit='s', utc=True)
            df_dia = df_dia.set_index('datetime')
            df_dia.index = df_dia.index.tz_convert('America/Sao_Paulo')
            
            # Adiciona o DataFrame do dia à nossa lista
            lista_dfs.append(df_dia)
            print(f"Coletado {len(df_dia)} candles para {hist_date.strftime('%Y-%m-%d')}")

    # Vai para o próximo dia
    hist_date += timedelta(days=1)

# --- 4. Consolidação e Processamento Final ---
if not lista_dfs:
    print("Nenhum dado foi coletado. Verifique o período e o ativo.")
else:
    print("\nConsolidando todos os dados coletados...")
    # Concatena todos os DataFrames diários em um só
    dados_completos = pd.concat(lista_dfs)
    
    # Remove duplicatas que podem surgir na junção dos dias
    dados_completos = dados_completos[~dados_completos.index.duplicated(keep='first')]
    dados_completos.sort_index(inplace=True)

    # Renomeia as colunas
    dados_completos.rename(columns={
        'open': 'abertura', 'high': 'maxima', 'low': 'minima',
        'close': 'fechamento', 'tick_volume': 'volume_ticks', 'real_volume': 'volume_real'
    }, inplace=True)
    
    # Seleciona as colunas de interesse
    dados_completos = dados_completos[['abertura', 'maxima', 'minima', 'fechamento', 'volume_ticks', 'volume_real']]

    print("\nFiltrando os dados pelo horário (10:00 às 12:59)...")
    mascara_horario = (dados_completos.index.hour >= HORA_INICIO_COLETA) & \
                      (dados_completos.index.hour < HORA_FIM_COLETA)
    dados_filtrados = dados_completos[mascara_horario]
    
    print(f"{len(dados_filtrados)} candles restaram após o filtro de horário.")

    # --- 5. Salvando em CSV ---
    if not dados_filtrados.empty:
        try:
            print(f"\nSalvando os dados filtrados no arquivo '{NOME_ARQUIVO_CSV}'...")
            dados_filtrados.to_csv(NOME_ARQUIVO_CSV)
            print("Arquivo salvo com sucesso!")
            print("\nAmostra dos dados salvos (verifique se o horário está correto):")
            print(dados_filtrados.head())
            print(dados_filtrados.tail())
        except Exception as e:
            print(f"Ocorreu um erro ao salvar o arquivo: {e}")
    else:
        print("Nenhum dado restou após a filtragem de horário.")

# --- 6. Encerramento da Conexão ---
print("\nEncerrando a conexão com o MetaTrader 5.")
mt5.shutdown()


Iniciando script de coleta de dados do Mini-Índice...
MetaTrader 5 inicializado com sucesso!
Versão: (500, 5495, '7 Jan 2026')
Iniciando loop de coleta para o ativo WIN$N...
Coletado 57 candles para 2025-01-02
Coletado 57 candles para 2025-01-03
Coletado 57 candles para 2025-01-06
Coletado 57 candles para 2025-01-07
Coletado 57 candles para 2025-01-08
Coletado 57 candles para 2025-01-09
Coletado 57 candles para 2025-01-10
Coletado 57 candles para 2025-01-13
Coletado 57 candles para 2025-01-14
Coletado 57 candles para 2025-01-15
Coletado 57 candles para 2025-01-16
Coletado 57 candles para 2025-01-17
Coletado 57 candles para 2025-01-20
Coletado 57 candles para 2025-01-21
Coletado 57 candles para 2025-01-22
Coletado 57 candles para 2025-01-23
Coletado 57 candles para 2025-01-24
Coletado 57 candles para 2025-01-27
Coletado 57 candles para 2025-01-28
Coletado 57 candles para 2025-01-29
Coletado 57 candles para 2025-01-30
Coletado 57 candles para 2025-01-31
Coletado 57 candles para 2025-02-0

True

In [21]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime, timedelta
import pytz
import os

print("Iniciando script de coleta de dados do Mini-Índice...")

# --- 1. Conexão com o MetaTrader 5 ---
if not mt5.initialize():
    print("Falha na inicialização do MetaTrader 5, erro:", mt5.last_error())
    quit()
else:
    print("MetaTrader 5 inicializado com sucesso!")
    print("Versão:", mt5.version())

# --- 2. Configurações da Coleta ---
ATIVO = 'WIN$N' 
TIMEFRAME = mt5.TIMEFRAME_M10

# Fuso horário de referência para a requisição
FUSO_REQUISICAO = pytz.timezone("Etc/UTC")

# Datas de início e fim do loop
data_inicio_loop = datetime(2025, 1, 1)
data_fim_loop = datetime.now()

# Nome do arquivo de saída atualizado
NOME_ARQUIVO_CSV = 'dados_mini_indice_pregao_completo.csv'

# Lista para armazenar os DataFrames de cada dia
lista_dfs = []

# --- 3. Loop de Coleta Diária ---

if not mt5.symbol_select(ATIVO, True):
    print(f"Ativo não pode ser selecionado: {ATIVO}")
    mt5.shutdown()
    quit()

print(f"Iniciando loop de coleta para o ativo {ATIVO}...")
hist_date = data_inicio_loop

while hist_date <= data_fim_loop:
    # Pula fins de semana
    if hist_date.weekday() not in (5, 6):
        
        # Define o início e o fim do dia no fuso UTC para a requisição
        utc_from = FUSO_REQUISICAO.localize(datetime(hist_date.year, hist_date.month, hist_date.day, hour=9))
        utc_to = FUSO_REQUISICAO.localize(datetime(hist_date.year, hist_date.month, hist_date.day, hour=22))
        
        rates = mt5.copy_rates_range(ATIVO, TIMEFRAME, utc_from, utc_to)
        
        if rates is not None and len(rates) > 0:
            df_dia = pd.DataFrame(rates)
            
            df_dia['datetime'] = pd.to_datetime(df_dia['time'], unit='s', utc=True)
            df_dia = df_dia.set_index('datetime')
            df_dia.index = df_dia.index.tz_convert('America/Sao_Paulo')
            
            lista_dfs.append(df_dia)
            print(f"Coletado {len(df_dia)} candles para {hist_date.strftime('%Y-%m-%d')}")

    hist_date += timedelta(days=1)

# --- 4. Consolidação e Processamento Final (COM A REMOÇÃO DO FILTRO) ---
if not lista_dfs:
    print("Nenhum dado foi coletado. Verifique o período e o ativo.")
else:
    print("\nConsolidando todos os dados coletados...")
    dados_completos = pd.concat(lista_dfs)
    
    dados_completos = dados_completos[~dados_completos.index.duplicated(keep='first')]
    dados_completos.sort_index(inplace=True)

    dados_completos.rename(columns={
        'open': 'abertura', 'high': 'maxima', 'low': 'minima',
        'close': 'fechamento', 'tick_volume': 'volume_ticks', 'real_volume': 'volume_real'
    }, inplace=True)
    
    dados_completos = dados_completos[['abertura', 'maxima', 'minima', 'fechamento', 'volume_ticks', 'volume_real']]

    # --- LÓGICA DE FILTRAGEM REMOVIDA ---
    # A variável 'dados_completos' agora contém todos os candles do pregão
    # e será salva diretamente.
    print(f"\nTotal de {len(dados_completos)} candles do pregão completo coletados.")

    # --- 5. Salvando em CSV ---
    try:
        print(f"\nSalvando os dados do pregão completo no arquivo '{NOME_ARQUIVO_CSV}'...")
        dados_completos.to_csv(NOME_ARQUIVO_CSV)
        print("Arquivo salvo com sucesso!")
        print("\nAmostra dos dados salvos (pregão completo):")
        print(dados_completos.head())
        print(dados_completos.tail())
    except Exception as e:
        print(f"Ocorreu um erro ao salvar o arquivo: {e}")

# --- 6. Encerramento da Conexão ---
print("\nEncerrando a conexão com o MetaTrader 5.")
mt5.shutdown()


Iniciando script de coleta de dados do Mini-Índice...
MetaTrader 5 inicializado com sucesso!
Versão: (500, 5495, '7 Jan 2026')
Iniciando loop de coleta para o ativo WIN$N...
Coletado 57 candles para 2025-01-02
Coletado 57 candles para 2025-01-03
Coletado 57 candles para 2025-01-06
Coletado 57 candles para 2025-01-07
Coletado 57 candles para 2025-01-08
Coletado 57 candles para 2025-01-09
Coletado 57 candles para 2025-01-10
Coletado 57 candles para 2025-01-13
Coletado 57 candles para 2025-01-14
Coletado 57 candles para 2025-01-15
Coletado 57 candles para 2025-01-16
Coletado 57 candles para 2025-01-17
Coletado 57 candles para 2025-01-20
Coletado 57 candles para 2025-01-21
Coletado 57 candles para 2025-01-22
Coletado 57 candles para 2025-01-23
Coletado 57 candles para 2025-01-24
Coletado 57 candles para 2025-01-27
Coletado 57 candles para 2025-01-28
Coletado 57 candles para 2025-01-29
Coletado 57 candles para 2025-01-30
Coletado 57 candles para 2025-01-31
Coletado 57 candles para 2025-02-0

True

In [34]:
import pandas as pd
import MetaTrader5 as mt5
from datetime import datetime, time, timedelta
import pytz

# 1. Inicializar o MetaTrader 5
if not mt5.initialize():
    print("Falha na inicialização do MetaTrader 5")
    mt5.shutdown()
    quit()

# Configurar o símbolo
symbol = "WIN$N"
timeframe = mt5.TIMEFRAME_M10  # 10 minutos

# Fuso horário UTC
utc_tz = pytz.UTC

# 2. DEFINIR A DATA DE INÍCIO NO FORMATO DIA, MÊS, ANO
# Exemplo: 1 de outubro de 2024
dia_inicio = 1    # Dia
mes_inicio = 10   # Mês
ano_inicio = 2024 # Ano

# 3. Definir período de coleta em UTC
# Criar datetime em UTC
start_date_utc = datetime(ano_inicio, mes_inicio, dia_inicio, 0, 0, 0, tzinfo=utc_tz)
end_date_utc = datetime.now(utc_tz)

print(f"Período UTC para coleta: {start_date_utc} a {end_date_utc}")
print(f"Isto corresponde em Brasília: {(start_date_utc - timedelta(hours=3)).strftime('%d/%m/%Y %H:%M')} a {(end_date_utc - timedelta(hours=3)).strftime('%d/%m/%Y %H:%M')}")

# 4. Coletar dados históricos (já em UTC)
print(f"\nColetando dados...")
rates = mt5.copy_rates_range(symbol, timeframe, start_date_utc, end_date_utc)

if rates is None or len(rates) == 0:
    print("Nenhum dado coletado. Verifique o símbolo e período.")
    mt5.shutdown()
    quit()

# 5. Converter para DataFrame
df = pd.DataFrame(rates)

# Converter timestamp UNIX para datetime UTC
df['time_utc'] = pd.to_datetime(df['time'], unit='s', utc=True)

# 6. Filtrar apenas dias úteis (segunda a sexta) e horário das 10h00 às 12h50 UTC
df['date_only'] = df['time_utc'].dt.date
df['hour_utc'] = df['time_utc'].dt.hour
df['minute_utc'] = df['time_utc'].dt.minute
df['day_of_week'] = df['time_utc'].dt.dayofweek  # 0=segunda, 6=domingo

# Filtrar: dias úteis (0-4) e horário entre 10:00 e 12:50 UTC
filtered_df = df[
    (df['day_of_week'] < 5) &  # Segunda a sexta
    (df['hour_utc'] >= 10) & 
    ((df['hour_utc'] < 12) | ((df['hour_utc'] == 12) & (df['minute_utc'] <= 50)))  # Até 12:50 inclusive
].copy()

print(f"\nTotal de candles coletados: {len(df)}")
print(f"Candles após filtro (10h-12h50 UTC, dias úteis): {len(filtered_df)}")

# 7. Selecionar e renomear colunas
columns_map = {
    'time_utc': 'Data_Hora_UTC',
    'open': 'Abertura',
    'high': 'Maxima',
    'low': 'Minima',
    'close': 'Fechamento',
    'tick_volume': 'Volume_Tick',
    'real_volume': 'Volume_Real'
}

# Nota: O volume real pode não estar disponível para todos os ativos
if 'real_volume' not in filtered_df.columns:
    filtered_df['real_volume'] = 0

filtered_df = filtered_df.rename(columns=columns_map)

# Adicionar horário de Brasília para referência (UTC-3)
filtered_df['Data_Hora_BR'] = filtered_df['Data_Hora_UTC'] - pd.Timedelta(hours=3)

# Ordenar colunas
colunas_finais = ['Data_Hora_UTC', 'Data_Hora_BR', 'Abertura', 'Maxima', 'Minima', 'Fechamento', 'Volume_Tick', 'Volume_Real']
filtered_df = filtered_df[colunas_finais]

# 8. Ordenar por data UTC
filtered_df.sort_values('Data_Hora_UTC', inplace=True)

# 9. Salvar em CSV
filename = f"WIN_10min_UTC_{start_date_utc.strftime('%Y%m%d')}_a_{end_date_utc.strftime('%Y%m%d')}.csv"
filtered_df.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"\n✅ Dados salvos em: {filename}")
print(f"📊 Total de candles: {len(filtered_df)}")

# 10. Mostrar estatísticas detalhadas
print("\n📈 ESTATÍSTICAS DETALHADAS:")
print("=" * 60)

# Período total
print(f"Período UTC completo: {filtered_df['Data_Hora_UTC'].min()} a {filtered_df['Data_Hora_UTC'].max()}")
print(f"Período BR correspondente: {filtered_df['Data_Hora_BR'].min()} a {filtered_df['Data_Hora_BR'].max()}")

# Dias úteis cobertos
dias_coletados = filtered_df['Data_Hora_UTC'].dt.date.nunique()
print(f"\n📅 Dias úteis com dados: {dias_coletados}")

# Distribuição por dia da semana
print(f"\n📊 Distribuição por dia da semana (UTC):")
dias_semana = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta']
for i in range(5):
    count = len(filtered_df[filtered_df['Data_Hora_UTC'].dt.dayofweek == i])
    if dias_coletados > 0:
        media = count / (dias_coletados / 5)  # Estimativa
        print(f"  {dias_semana[i]}: {count} candles (~{media:.1f} por dia)")
    else:
        print(f"  {dias_semana[i]}: {count} candles")

# Distribuição por horário UTC
print(f"\n⏰ Distribuição por horário (UTC):")
horarios_utc = {}
for hora in range(10, 13):
    candles_hora = filtered_df[filtered_df['Data_Hora_UTC'].dt.hour == hora]
    if len(candles_hora) > 0:
        horarios_utc[hora] = len(candles_hora)
        # Detalhar por minutos
        minutos = candles_hora['Data_Hora_UTC'].dt.minute.unique()
        minutos.sort()
        minutos_str = ', '.join([f"{m:02d}" for m in minutos])
        print(f"  {hora:02d}:XX: {len(candles_hora)} candles (minutos: {minutos_str})")
    else:
        print(f"  {hora:02d}:XX: 0 candles")

# Primeiras e últimas velas
print(f"\n🔍 Primeiros 3 candles:")
print("-" * 100)
for idx, row in filtered_df.head(3).iterrows():
    print(f"UTC: {row['Data_Hora_UTC'].strftime('%Y-%m-%d %H:%M')} | "
          f"BR: {row['Data_Hora_BR'].strftime('%H:%M')} | "
          f"Abertura: {row['Abertura']:.0f} | Fechamento: {row['Fechamento']:.0f} | "
          f"Variação: {(row['Fechamento']-row['Abertura']):+.0f}")

print(f"\n🔍 Últimos 3 candles:")
print("-" * 100)
for idx, row in filtered_df.tail(3).iterrows():
    print(f"UTC: {row['Data_Hora_UTC'].strftime('%Y-%m-%d %H:%M')} | "
          f"BR: {row['Data_Hora_BR'].strftime('%H:%M')} | "
          f"Abertura: {row['Abertura']:.0f} | Fechamento: {row['Fechamento']:.0f} | "
          f"Variação: {(row['Fechamento']-row['Abertura']):+.0f}")

# 11. Encerrar conexão
mt5.shutdown()

print(f"\n💾 Arquivo salvo com sucesso!")
print(f"📍 Caminho: {filename}")
print("\n⚠️  NOTA: Os dados foram filtrados para horário UTC das 10:00 às 12:50")
print("   (Em Brasília, isso corresponde a 07:00 às 09:50 durante o horário padrão)")

Período UTC para coleta: 2024-10-01 00:00:00+00:00 a 2026-01-15 03:48:17.269112+00:00
Isto corresponde em Brasília: 30/09/2024 21:00 a 15/01/2026 00:48

Coletando dados...

Total de candles coletados: 18206
Candles após filtro (10h-12h50 UTC, dias úteis): 5740

✅ Dados salvos em: WIN_10min_UTC_20241001_a_20260115.csv
📊 Total de candles: 5740

📈 ESTATÍSTICAS DETALHADAS:
Período UTC completo: 2024-10-01 10:00:00+00:00 a 2026-01-14 12:50:00+00:00
Período BR correspondente: 2024-10-01 07:00:00+00:00 a 2026-01-14 09:50:00+00:00

📅 Dias úteis com dados: 319

📊 Distribuição por dia da semana (UTC):
  Segunda: 1170 candles (~18.3 por dia)
  Terça: 1170 candles (~18.3 por dia)
  Quarta: 1116 candles (~17.5 por dia)
  Quinta: 1114 candles (~17.5 por dia)
  Sexta: 1170 candles (~18.3 por dia)

⏰ Distribuição por horário (UTC):
  10:XX: 1912 candles (minutos: 00, 10, 20, 30, 40, 50)
  11:XX: 1914 candles (minutos: 00, 10, 20, 30, 40, 50)
  12:XX: 1914 candles (minutos: 00, 10, 20, 30, 40, 50)

🔍 P

# RASCUNHO

*****************************
* vale a pena Explorar outros indicadores úteis, como Bandas de Bollinger ou Volume Profile?


* Como definir os limiares para as classes "Compra", "Venda" e "Lateralidade".

* Qauis técnicas de regularização usar?

* Discutir como monitorar a perda (loss) durante o treinamento para diagnosticar problemas.

******************************

*ivar (maior variação de preço do dia das 9h às 16h)
*ivarTreino (maior variação de preço do dia das 9h às 17h)
*tam (tamanho da vela) --------------------------------> valor de fechamento - valor de abertura (indica o tamanho da vela, valor positivo=verde, valor negativo=vermelho)
*tamNor (normalização do tamanho da vela) -------------> tam/ivar 
*tamNorTreino (normalização do tamanho da vela) -------------> tam/ivarTreino 
*#cor (cor da vela) ------------------------------------> tam/|tam|	(+1=verde, -1=vermelho), talves essa variável esteja correlacionada com o tamanho da vela
*var (variação de preço) ------------------------------> preço máximo da vela - preço mínimo da vela  (indica o tamanho da variação entre max e min daquela hora, é o tamanho do pavío + tamanho da vela)
*varNor (normalização da variação de preço) -----------> var/ivar  
*varMax (maior preço de negociação do dia)
*preco (valor de abertura normalizado) ----------------> valor de abertura/varMax 
*vol (volume de ordens)
*ivol (maior volume encontrado no dia, das 9h às 16h)
*vol/ivolTreino (maior volume encontrado no dia, das 9h às 17h)
*volNor (volume normalizado) --------------------------> vol/ivol 
*volNorTreino (volume normalizado) --------------------------> vol/ivolTreino 

-----------VARIÁVEIS TRATADAS PARA TREINO-------------------------------
tamMed (é o tamanho médio entre as velas do dia) ------------> (soma de todas as tamNor)/(quantidade de velas do dia) 	 
volMed (é o volume médio normalizado entre as velas do dia) -> (soma de todos os volNor do dia)/(quantidade de velasdo dia)		

-----------VARIÁVEIS TRATADAS PARA ENTRADA-------------------------------
v1 -> valor de abertura normalizado
v2 -> tamNor (tamanho da vela para aquele horário,  normalizado das 9h às 16h)
v3 -> varNor (variação de preço para aquele horário, normalizado das 9h às 16h)
v4 -> volNor (volume normalizado para aquele horário,  normalizado das 9h às 16h)


-----------------VARIÁVEIS DE ENTRADA------------------
x1 a x5   dados da vela das 9h
x6 a x10  dados da vela das 10h
x11 a x15   dados da vela das 11h
x16 a x20  dados da vela das 12h
x21 a x25   dados da vela das 13h
x26 a x30  dados da vela das 14h
x31 a x35   dados da vela das 15h
x36 a x40   dados da vela das 16h



----------------VARIÁVEIS DE SAIDA--------------
y1 -> compra
y2 -> lateral
y3 -> venda


-----------VARIÁVEIS TRATADAS PARA TREINO (COMPRA)-----------------
VELA VERDE: v2 = +1
VELA GRANDE: tamNorTreino >= [ 50% * tamMed ]
ALTO VOLUME: volNorTreino >= [ 50% * volMed ]

-----------VARIÁVEIS TRATADAS PARA TREINO (LATERAL)-----------------
VELA PEQUENA: tamNorTreino<= [ 50% * tamMed ]
PEQUENO VOLUME: volNor <= [ 50% * volMed ]

-----------VARIÁVEIS TRATADAS PARA TREINO (venda)-----------------
VELA VERMELHA: v2 = -1
VELA GRANDE: tamNorTreino>= [ 50% * tamMed ]
ALTO VOLUME: volNorTreino>= [ 50% * volMed ]